In [ ]:
# Cell 1: Inspect the runtime and available hardware

import sys
from importlib.metadata import version, PackageNotFoundError

# Check Python and installed packages before choosing dependency versions.
print("Python:", sys.version.split()[0])

for package in ("torch", "torchvision", "transformers", "numpy", "Pillow"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

# Use a GPU when available. Otherwise, select the CPU.
try:
    import torch
except ImportError:
    DEVICE = "cpu"
    print("PyTorch needs to be installed in the next cell.")
else:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

print("Selected device:", DEVICE)
print("Runtime inspection complete.")

In [ ]:
# Cell 2: Install and verify the OCR dependencies

import sys
import subprocess
import time

started = time.perf_counter()

# Retain Colab's installed PyTorch and CUDA.
# Install the Transformers version used by this notebook.
subprocess.check_call([
    sys.executable,
    "-m", "pip", "install", "--quiet",
    "transformers==5.17.0",
    "sentencepiece==0.2.1",
    "sacremoses==0.1.1",
])

# Verify the recognition APIs after installation before loading model weights.
import torch
import transformers
from transformers import (
    RobertaTokenizer,
    ViTImageProcessor,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoImageProcessor,
    AutoModelForTextRecognition,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Selected device:", DEVICE)

# Check GPU memory and execute a small tensor operation to verify CUDA access.
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {memory_gb:.1f} GB")
    print("GPU tensor check:", torch.ones(1, device=DEVICE).item())
else:
    print("Select the A100 GPU before running the training cells.")

print("OCR imports verified.")
print(f"Cell 2 completed in {time.perf_counter() - started:.1f} seconds.")

In [ ]:
# Cell 3: Load the pretrained English TrOCR recogniser

import time
import torch
from huggingface_hub import hf_hub_download
from transformers import (
    RobertaTokenizer,
    ViTImageProcessor,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
)

started = time.perf_counter()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENGLISH_MODEL_ID = "microsoft/trocr-base-handwritten"

# Build the tokenizer from this checkpoint's own files.
vocab_file = hf_hub_download(ENGLISH_MODEL_ID, "vocab.json")
merges_file = hf_hub_download(ENGLISH_MODEL_ID, "merges.txt")

english_tokenizer = RobertaTokenizer(
    vocab=vocab_file,
    merges=merges_file,
    model_max_length=512,
)

# Check tokenisation before downloading the model weights.
test_text = "Hello world"
test_ids = english_tokenizer.encode(test_text)

if test_ids != [0, 31414, 232, 2]:
    raise RuntimeError(f"Unexpected tokenizer output: {test_ids}")

print(
    "Tokenizer check:",
    english_tokenizer.decode(test_ids, skip_special_tokens=True),
)

# Use the checkpoint's image preprocessing settings.
english_image_processor = ViTImageProcessor.from_pretrained(
    ENGLISH_MODEL_ID
)

# Combine the image processor and verified tokenizer.
english_processor = TrOCRProcessor(
    image_processor=english_image_processor,
    tokenizer=english_tokenizer,
)

print("Loading English model weights...")

english_model = VisionEncoderDecoderModel.from_pretrained(
    ENGLISH_MODEL_ID,
    dtype=torch.float32,
).to(DEVICE)

# Use inference mode settings; this cell loads the baseline without training it.
english_model.eval()

if DEVICE == "cuda":
    torch.cuda.synchronize()

print("English recogniser loaded.")
print("Model device:", english_model.device)
print("Model precision:", english_model.dtype)
print("Training mode:", english_model.training)
print(f"Cell 3 completed in {time.perf_counter() - started:.1f} seconds.")

In [ ]:
# Cell 4: Connect Google Drive and prepare project folders

from pathlib import Path
from google.colab import drive

# Keep data and training checkpoints across Colab sessions.
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/handwritten_translation_project"
)

# Separate collected data, public data, saved model weights, and evaluation results.
GROUP_DATA_DIR = PROJECT_DIR / "data" / "group"
PUBLIC_DATA_DIR = PROJECT_DIR / "data" / "public"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULTS_DIR = PROJECT_DIR / "results"

for folder in (
    GROUP_DATA_DIR,
    PUBLIC_DATA_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
):
    folder.mkdir(parents=True, exist_ok=True)
    print("Ready:", folder)

print("Project folders are ready.")

In [ ]:
# Cell 4A: Verify the saved dataset and cache it locally
import csv, hashlib, io, json, shutil
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

CURATED_ROOT = Path(
    '/content/drive/MyDrive/CVDL_Handwriting_Dataset/v1_writers'
)
WRITERS = {
    'W01': ('Kim', 'validation'),
    'W02': ('Reiner', 'test'),
    'W03': ('Yu Wei', 'test'),
    'W04': ('Wei Kang', 'train'),
    'W05': ('Wei En', 'test'),
}
FIELDS = (
    'sample_id', 'image_path', 'text', 'language', 'source_page_id',
    'capture_id', 'line_index', 'writer_id', 'writer_name',
)
SPLITS = ('train', 'validation', 'test')
CURATED_READY = False


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def fingerprint(rows):
    # Local paths are excluded. Adding test data does not alter the train ID.
    values = [
        {k: r[k] for k in (*FIELDS, 'split', 'sha256')}
        for r in rows
    ]
    values.sort(key=lambda r: (r['split'], r['sample_id']))
    return hashlib.sha256(
        json.dumps(
            values, ensure_ascii=False, sort_keys=True
        ).encode()
    ).hexdigest()


def prepare_curated(root):
    root = root.resolve()
    problems, rows, manifests, capture_lines = [], [], {}, {}
    indexes = {
        k: defaultdict(list)
        for k in (
            'sample_id', 'path', 'pixels',
            'writer_id', 'source_page_id', 'capture_id',
        )
    }

    meta = read_json(root / 'metadata/dataset.json')
    expected = {
        w: {'name': name, 'split': split}
        for w, (name, split) in WRITERS.items()
    }
    if (
        meta.get('writers') != expected
        or meta.get('collection_mode') != 'writer_assigned'
    ):
        problems.append(
            'metadata/dataset.json disagrees with the writer assignments.'
        )

    captures = read_json(root / 'captures.json')
    if not isinstance(captures, list):
        raise ValueError(
            'captures.json must be a list, as in the inspected dataset.'
        )

    capture_ids = set()
    for cap in captures:
        split = cap.get('split')
        if split == 'pending':
            continue

        cid, writer = cap.get('capture_id'), cap.get('writer_id')
        if cid in capture_ids or split not in SPLITS:
            problems.append(f'Duplicate capture or invalid split: {cid}')
        capture_ids.add(cid)

        if WRITERS.get(writer) != (cap.get('writer_name'), split):
            problems.append(f'Capture writer assignment mismatch: {cid}')

        try:
            record = read_json(
                root / split / 'records' / f'{cid}.json'
            )
            # Compare active fields; historical migration fields are ignored.
            for key in (
                'capture_id', 'source_page_id', 'split',
                'writer_id', 'writer_name', 'language', 'lines',
            ):
                if record.get(key) != cap.get(key):
                    problems.append(
                        f'Capture record/index mismatch: {cid}, {key}'
                    )
        except Exception as e:
            problems.append(f'Cannot read capture record {cid}: {e}')

        for line in cap.get('lines', []):
            sid = line.get('sample_id')
            if sid in capture_lines:
                problems.append(
                    f'Duplicate sample in capture index: {sid}'
                )
            capture_lines[sid] = (split, line)

            for key in (
                'capture_id', 'source_page_id',
                'writer_id', 'writer_name', 'language',
            ):
                if line.get(key) != cap.get(key):
                    problems.append(
                        f'Capture/line mismatch: {sid}, {key}'
                    )

    for split in SPLITS:
        path = root / split / 'labels.csv'
        if not path.exists():
            print(f'{split}: manifest absent; split unavailable.')
            continue

        manifests[split] = path.read_bytes()
        reader = csv.DictReader(
            io.StringIO(manifests[split].decode('utf-8-sig'))
        )
        if not set(FIELDS).issubset(reader.fieldnames or []):
            problems.append(f'{split}: missing required CSV columns.')
            continue

        for line_no, r in enumerate(reader, 2):
            tag = f'{split}/labels.csv:{line_no}'
            if None in r or any(
                not isinstance(r.get(k), str) or not r[k].strip()
                for k in FIELDS
            ):
                problems.append(
                    f'{tag}: malformed row or empty field/label.'
                )
                continue

            r = {**r, 'split': split}
            rows.append(r)  # Preserve corrected text exactly.

            if WRITERS.get(r['writer_id']) != (r['writer_name'], split):
                problems.append(f'{tag}: writer assignment/name mismatch.')
            if (
                r['language'] not in ('en', 'zh')
                or not r['line_index'].isdigit()
            ):
                problems.append(f'{tag}: invalid language or line index.')

            saved = capture_lines.get(r['sample_id'])
            if (
                not saved
                or saved[0] != split
                or any(
                    str(saved[1].get(k, '')) != r[k]
                    for k in FIELDS
                )
            ):
                problems.append(f'{tag}: CSV differs from capture index.')

            for key in (
                'sample_id', 'writer_id', 'source_page_id', 'capture_id'
            ):
                indexes[key][r[key]].append(split)

            try:
                # Inspection confirmed dataset-root-relative paths.
                relative = Path(r['image_path'])
                source = (root / relative).resolve()
                expected_folder = (
                    root / split / 'images' / r['writer_id']
                )
                if (
                    relative.is_absolute()
                    or '..' in relative.parts
                    or not source.is_relative_to(expected_folder)
                ):
                    raise ValueError(
                        'image_path is outside its assigned image folder'
                    )

                raw = source.read_bytes()
                r['sha256'] = hashlib.sha256(raw).hexdigest()
                if (
                    saved
                    and saved[1].get('image_sha256') != r['sha256']
                ):
                    problems.append(
                        f'{tag}: image hash differs from saved record.'
                    )

                with Image.open(io.BytesIO(raw)) as im:
                    rgb = im.convert('RGB')
                    pixels = hashlib.sha256(
                        str(rgb.size).encode() + rgb.tobytes()
                    ).hexdigest()

                indexes['path'][str(source)].append(split)
                indexes['pixels'][pixels].append(split)

                if len(rows) % 100 == 0:
                    print(f'Checked {len(rows)} line images')

            except Exception as e:
                problems.append(
                    f'{tag}: missing/unreadable/invalid image: {e}'
                )

    if set(capture_lines) != {r['sample_id'] for r in rows}:
        problems.append(
            'Manifest sample IDs and approved capture-index sample IDs differ.'
        )

    for key, mapping in indexes.items():
        for value, splits in mapping.items():
            duplicate = (
                key in ('sample_id', 'path', 'pixels')
                and len(splits) > 1
            )
            if duplicate or len(set(splits)) > 1:
                problems.append(
                    f'Duplicate or cross-split {key}: '
                    f'{value}, {sorted(set(splits))}'
                )

    for split in SPLITS:
        subset = [r for r in rows if r['split'] == split]
        pages = len({r['source_page_id'] for r in subset})
        print(f'{split}: {len(subset)} lines, {pages} pages')
        print(
            '  Writer/language:',
            dict(Counter(
                (r['writer_id'], r['language']) for r in subset
            )),
        )

    if problems:
        print('\n'.join(problems[:50]))
        raise ValueError(
            f'{len(problems)} dataset problems. '
            'Source unchanged; training blocked.'
        )

    cache = Path('/content/cvdl_curated_cache') / fingerprint(rows)[:16]

    for split, raw in manifests.items():
        destination = cache / split / 'labels.csv'
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(raw)

    for i, r in enumerate(rows, 1):
        destination = cache / r['image_path']
        destination.parent.mkdir(parents=True, exist_ok=True)

        if (
            not destination.exists()
            or hashlib.sha256(
                destination.read_bytes()
            ).hexdigest() != r['sha256']
        ):
            shutil.copyfile(
                root / r['image_path'], destination
            )

        if hashlib.sha256(
            destination.read_bytes()
        ).hexdigest() != r['sha256']:
            raise ValueError(
                f'Copy changed during audit: {r["sample_id"]}; '
                'rerun this cell.'
            )

        r['local_path'] = str(destination)
        if i % 100 == 0:
            print(f'Cached {i}/{len(rows)} images')

    print(
        'Verified. Local cache:', cache,
        '| Pending excluded; source unchanged.'
    )
    return rows


CURATED_ROWS = prepare_curated(CURATED_ROOT)
CURATED_READY = True

In [ ]:
# Cell 5: Inspect the IAM and CASIA dataset metadata

%pip -q install "datasets==4.8.5"

import datasets
from datasets import load_dataset_builder

PUBLIC_DATASETS = {
    "English": "Teklia/IAM-line",
    "Chinese": "Teklia/CASIA-HWDB2-line",
}

print("Datasets library:", datasets.__version__)

# Inspect the image/text schema and published splits before selecting any samples.
for language, dataset_id in PUBLIC_DATASETS.items():
    info = load_dataset_builder(dataset_id).info

    print(f"\n{language}: {dataset_id}")
    print("Fields:", list(info.features))
    print(
        "Splits:",
        {name: split.num_examples for name, split in info.splits.items()}
    )

    # Stop early if a dataset lacks the fields or splits required by later cells.
    assert {"image", "text"}.issubset(info.features)
    assert {"train", "validation", "test"}.issubset(info.splits)

print("\nPublic dataset metadata verified.")

In [ ]:
# Cell 6: Prepare and cache the public dataset subsets

from datasets import Dataset, load_dataset, load_from_disk

# Use small, separate training and validation subsets for the initial prototype.
SAMPLE_COUNTS = {"train": 512, "validation": 64}
PUBLIC_SUBSETS = {}

for language, dataset_id in PUBLIC_DATASETS.items():
    PUBLIC_SUBSETS[language] = {}

    for split, count in SAMPLE_COUNTS.items():
        folder = PUBLIC_DATA_DIR / f"{language.lower()}_{split}_{count}"

        # Reuse the saved subset so later sessions evaluate the same selected lines.
        if (folder / "state.json").exists():
            subset = load_from_disk(str(folder))
            source = "loaded from Drive"
        else:
            # Stream the requested split and save only the selected subset to Drive.
            stream = load_dataset(dataset_id, split=split, streaming=True)

            # Shuffle training examples reproducibly before selecting them.
            if split == "train":
                stream = stream.shuffle(seed=42, buffer_size=1024)

            rows = list(stream.take(count))
            subset = Dataset.from_list(rows, features=stream.features)
            subset.save_to_disk(str(folder))
            source = "downloaded and saved"

        assert len(subset) == count
        assert {"image", "text"}.issubset(subset.column_names)

        PUBLIC_SUBSETS[language][split] = subset
        print(f"{language} {split}: {len(subset)} lines ({source})")

print("\nTraining and validation subsets are ready. Test splits were not loaded.")

In [ ]:
# Cell 6A: shared scoring and Chinese augmentation helpers
import importlib.util
from pathlib import Path

# Embedded script: no separate upload is needed.
shared_script = Path.cwd() / 'scripts' / 'ocr_workflow.py'
shared_script.parent.mkdir(parents=True, exist_ok=True)
shared_script.write_text('"""Shared OCR scoring and training-only image augmentation.\n\nCLI: python ocr_workflow.py results.json --language zh\nAccepts {references: [...], predictions: [...]} or a JSON list of\n{reference: ..., prediction: ...} records. Scores are corpus-level ratios.\n"""\nimport argparse\nimport hashlib\nimport json\nimport random\nimport unicodedata\nfrom pathlib import Path\n\nMETRIC_POLICY = \'ocr-v2-raw-nfkc-punctuation\'\n# Explicit equivalences; preserve punctuation rather than deleting it.\nPUNCTUATION = str.maketrans({\n    \'。\': \'.\', \'，\': \',\', \'、\': \',\', \'；\': \';\', \'：\': \':\',\n    \'！\': \'!\', \'？\': \'?\', \'“\': \'"\', \'”\': \'"\', \'‘\': "\'", \'’\': "\'",\n    \'「\': \'"\', \'」\': \'"\', \'『\': \'"\', \'』\': \'"\',\n    \'【\': \'[\', \'】\': \']\', \'—\': \'-\', \'–\': \'-\', \'…\': \'...\',\n})\nAUGMENTATION = dict(enabled=True, probability=0.7, rotation_degrees=1.5,\n                    brightness=(0.9, 1.1), contrast=(0.85, 1.15),\n                    blur_probability=0.2, blur_radius=(0.1, 0.5), seed=42)\n\n\ndef normalize_punctuation(text):\n    """Evaluation only: NFKC then the explicit punctuation table above.\n\n    Preserve case and whitespace; do not convert simplified/traditional Chinese.\n    """\n    return unicodedata.normalize(\'NFKC\', text).translate(PUNCTUATION)\n\n\ndef distance(a, b):\n    previous = list(range(len(b) + 1))\n    for i, x in enumerate(a, 1):\n        current = [i]\n        for j, y in enumerate(b, 1):\n            current.append(min(current[-1] + 1, previous[j] + 1,\n                               previous[j - 1] + (x != y)))\n        previous = current\n    return previous[-1]\n\n\ndef error_rates(references, predictions, lang):\n    references, predictions = list(references), list(predictions)\n    if lang not in (\'en\', \'zh\'):\n        raise ValueError(\'language must be en or zh\')\n    if not references or len(references) != len(predictions):\n        raise ValueError(\'Need nonempty, equally sized reference/prediction lists\')\n    if not all(isinstance(x, str) for x in references + predictions):\n        raise TypeError(\'Every reference and prediction must be a string\')\n\n    def rate(refs, preds):\n        total = sum(map(len, refs))\n        # Undefined when all reference strings/word lists are empty.\n        return sum(distance(r, p) for r, p in zip(refs, preds)) / total if total else None\n\n    nfkc = lambda xs: [unicodedata.normalize(\'NFKC\', x) for x in xs]\n    normal = lambda xs: [normalize_punctuation(x) for x in xs]\n    return dict(CER=rate(references, predictions),\n                CER_NFKC=rate(nfkc(references), nfkc(predictions)),\n                CER_PUNCT=rate(normal(references), normal(predictions)),\n                WER=rate([x.split() for x in references],\n                         [x.split() for x in predictions]) if lang == \'en\' else None,\n                exact_match=sum(r == p for r, p in zip(references, predictions)) / len(references))\n\n\ndef evaluate_rows(rows, predict, lang):\n    """One scorer for all model versions and both held-out splits.\n\n    predict(row) returns untouched decoded text. Never augment evaluation rows.\n    """\n    rows = list(rows)\n    refs = [r[\'text\'] for r in rows]\n    preds = [predict(r) for r in rows]\n    return dict(metric_policy=METRIC_POLICY, lines=len(rows),\n                references=refs, predictions=preds,\n                **error_rates(refs, preds, lang))\n\n\ndef augment_line(image, sample_key, epoch=0, config=None):\n    """Mild image-only changes, deterministic by sample/epoch; labels unchanged.\n\n    Rotation expands the canvas to avoid clipping end characters. No flips,\n    random crops, or erasing. Pillow is imported lazily for a dependency-free CLI.\n    """\n    from PIL import Image, ImageEnhance, ImageFilter\n    cfg = AUGMENTATION if config is None else config\n    image = image.convert(\'RGB\')\n    seed = hashlib.sha256(f"{cfg[\'seed\']}:{epoch}:{sample_key}".encode()).digest()\n    rng = random.Random(int.from_bytes(seed[:8], \'big\'))\n    if not cfg[\'enabled\'] or rng.random() >= cfg[\'probability\']:\n        return image\n    image = image.rotate(rng.uniform(-cfg[\'rotation_degrees\'], cfg[\'rotation_degrees\']),\n                         resample=Image.Resampling.BICUBIC, expand=True,\n                         fillcolor=(255, 255, 255))\n    image = ImageEnhance.Brightness(image).enhance(rng.uniform(*cfg[\'brightness\']))\n    image = ImageEnhance.Contrast(image).enhance(rng.uniform(*cfg[\'contrast\']))\n    if rng.random() < cfg[\'blur_probability\']:\n        image = image.filter(ImageFilter.GaussianBlur(rng.uniform(*cfg[\'blur_radius\'])))\n    return image\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'input\', type=Path)\n    parser.add_argument(\'--language\', choices=[\'en\', \'zh\'], required=True)\n    parser.add_argument(\'--output\', type=Path)\n    args = parser.parse_args()\n    data = json.loads(args.input.read_text(encoding=\'utf-8-sig\'))\n    if isinstance(data, list):\n        groups = {(r.get(\'split\'), r.get(\'language\'), r.get(\'model\')) for r in data}\n        if len(groups) > 1:\n            raise ValueError(\'Choose one split/language/model before scoring records\')\n        refs = [r[\'reference\'] for r in data]\n        preds = [r[\'prediction\'] for r in data]\n    else:\n        refs, preds = data[\'references\'], data[\'predictions\']\n    result = dict(metric_policy=METRIC_POLICY, language=args.language, lines=len(refs),\n                  **error_rates(refs, preds, args.language))\n    output = json.dumps(result, ensure_ascii=False, indent=2)\n    if args.output:\n        args.output.write_text(output + \'\\n\', encoding=\'utf-8\')\n    print(output)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
spec = importlib.util.spec_from_file_location('minimal_ocr_helpers', shared_script)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
error_rates = workflow.error_rates
evaluate_rows = workflow.evaluate_rows
augment_line = workflow.augment_line
AUGMENTATION = dict(workflow.AUGMENTATION)
print('Shared metrics:', workflow.METRIC_POLICY)
print('Chinese training augmentation:', AUGMENTATION)


In [ ]:
# Cell 7: Evaluate the English baseline

%pip -q install "jiwer==4.0.0"

import json
import torch
from jiwer import cer, wer

validation = PUBLIC_SUBSETS["English"]["validation"]
references = [text.strip() for text in validation["text"]]
predictions = []

english_model.eval()
batch_size = 8

# Generate transcriptions in batches without computing training gradients.
with torch.inference_mode():
    for start in range(0, len(validation), batch_size):
        batch = validation[start : start + batch_size]
        images = [image.convert("RGB") for image in batch["image"]]

        pixel_values = english_processor(
            images=images, return_tensors="pt"
        ).pixel_values.to(english_model.device)

        token_ids = english_model.generate(
            pixel_values, max_new_tokens=128
        )
        predictions.extend(
            text.strip()
            for text in english_processor.batch_decode(
                token_ids, skip_special_tokens=True
            )
        )
        print(f"Recognised {len(predictions)}/{len(validation)} lines")

# Record predictions and references with CER/WER for the later model comparison.
# Both metrics measure errors, so lower values are better.
english_baseline = {
    "model": "microsoft/trocr-base-handwritten",
    "dataset": PUBLIC_DATASETS["English"],
    "split": "validation",
    "references": references,
    "predictions": predictions,
    "cer": cer(references, predictions),
    "wer": wer(references, predictions),
}

output_path = RESULTS_DIR / "english_baseline_validation.json"
output_path.write_text(
    json.dumps(english_baseline, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"\nCharacter error rate: {english_baseline['cer']:.2%}")
print(f"Word error rate:      {english_baseline['wer']:.2%}")
print("First reference:     ", repr(references[0]))
print("First prediction:    ", repr(predictions[0]))
print("Results saved to:", output_path)

In [ ]:
# Cell 8: Load and check the Chinese recogniser

import torch
from transformers import AutoImageProcessor, AutoModelForTextRecognition

CHINESE_MODEL_ID = "PaddlePaddle/PP-OCRv5_server_rec_safetensors"

# Load the matching image processor and pretrained Chinese recognition weights.
chinese_processor = AutoImageProcessor.from_pretrained(CHINESE_MODEL_ID)
chinese_model = AutoModelForTextRecognition.from_pretrained(
    CHINESE_MODEL_ID, dtype=torch.float32
).to(DEVICE).eval()

# Use one validation line as a smoke check before evaluating the full subset.
sample = PUBLIC_SUBSETS["Chinese"]["validation"][0]
image = sample["image"].convert("RGB")

with torch.inference_mode():
    inputs = chinese_processor(
        images=image, return_tensors="pt"
    ).to(chinese_model.device)
    outputs = chinese_model(**inputs)

# Decode the model output into readable text using the model-specific processor.
result = chinese_processor.post_process_text_recognition(outputs)[0]

print("Model device:", chinese_model.device)
print("Reference:  ", repr(sample["text"]))
print("Prediction: ", repr(result["text"]))
print("Chinese recogniser loaded and checked.")

In [ ]:
# Cell 9: Evaluate the Chinese baseline

import json
import torch
import unicodedata
from jiwer import cer

validation = PUBLIC_SUBSETS["Chinese"]["validation"]
references = [text.strip() for text in validation["text"]]
predictions = []

chinese_model.eval()
batch_size = 8

# Transcribe each validation batch with the original pretrained Chinese model.
with torch.inference_mode():
    for start in range(0, len(validation), batch_size):
        batch = validation[start : start + batch_size]
        images = [image.convert("RGB") for image in batch["image"]]

        inputs = chinese_processor(
            images=images, return_tensors="pt"
        ).to(chinese_model.device)

        outputs = chinese_model(**inputs)
        results = chinese_processor.post_process_text_recognition(outputs)
        predictions.extend(result["text"].strip() for result in results)

        print(f"Recognised {len(predictions)}/{len(validation)} lines")

assert len(predictions) == len(references)

# Keep exact CER and NFKC-normalized CER separate. NFKC unifies compatible
# Unicode forms, such as full-width characters, without correcting recognition errors.
normalized_references = [
    unicodedata.normalize("NFKC", text) for text in references
]
normalized_predictions = [
    unicodedata.normalize("NFKC", text) for text in predictions
]

# Save both error measures and the underlying text pairs for later comparison.
chinese_baseline = {
    "model": CHINESE_MODEL_ID,
    "dataset": PUBLIC_DATASETS["Chinese"],
    "split": "validation",
    "references": references,
    "predictions": predictions,
    "cer": cer(references, predictions),
    "cer_nfkc": cer(normalized_references, normalized_predictions),
}

output_path = RESULTS_DIR / "chinese_baseline_validation.json"
output_path.write_text(
    json.dumps(chinese_baseline, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"\nExact-character CER: {chinese_baseline['cer']:.2%}")
print(f"NFKC-normalized CER: {chinese_baseline['cer_nfkc']:.2%}")
print("First reference:    ", repr(references[0]))
print("First prediction:   ", repr(predictions[0]))
print("Results saved to:", output_path)

In [ ]:
# Cell 10: Prepare English training batches

import torch
from torch.utils.data import DataLoader

# Align the decoder start, padding, and vocabulary settings with the tokenizer.
tokenizer = english_processor.tokenizer
english_model.config.decoder_start_token_id = tokenizer.eos_token_id
english_model.config.pad_token_id = tokenizer.pad_token_id
english_model.config.vocab_size = english_model.config.decoder.vocab_size

# Convert a list of image/text examples into model inputs and padded target labels.
def prepare_english_batch(rows):
    images = [row["image"].convert("RGB") for row in rows]
    texts = [row["text"] for row in rows]

    pixel_values = english_processor(
        images=images, return_tensors="pt"
    ).pixel_values

    tokens = tokenizer(texts, padding=True, return_tensors="pt")

    # The model supplies its own start token. Remove the tokenizer's <s>.
    labels = tokens.input_ids[:, 1:].clone()
    # Ignore padded target positions when the model calculates the training loss.
    labels[tokens.attention_mask[:, 1:] == 0] = -100

    return {"pixel_values": pixel_values, "labels": labels}

# Shuffle training examples and prepare batches of four using the collator above.
ENGLISH_BATCH_SIZE = 4
english_train_loader = DataLoader(
    PUBLIC_SUBSETS["English"]["train"],
    batch_size=ENGLISH_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=prepare_english_batch,
)

# Check tensor shapes and a finite loss before starting any weight updates.
check_batch = next(iter(english_train_loader))
english_model.eval()

with torch.inference_mode():
    check_loss = english_model(
        pixel_values=check_batch["pixel_values"].to(english_model.device),
        labels=check_batch["labels"].to(english_model.device),
    ).loss

assert torch.isfinite(check_loss).item()

print("Image shape:", tuple(check_batch["pixel_values"].shape))
print("Label shape:", tuple(check_batch["labels"].shape))
print("Batch loss:", round(check_loss.item(), 4))
print("English training batches are ready.")

In [ ]:
# Cell 11: Fine-tune the English recogniser for one epoch

import json
import torch

# This training loop requires CUDA and BF16 support; check both before starting.
assert english_model.device.type == "cuda"
assert torch.cuda.is_bf16_supported()

torch.manual_seed(42)
learning_rate = 5e-6
optimizer = torch.optim.AdamW(english_model.parameters(), lr=learning_rate)

english_model.train()
total_loss = 0.0
total_steps = len(english_train_loader)

print(f"Training on 512 lines for {total_steps} steps...")

# Process every training batch once, using BF16 autocast for the forward pass.
for step, batch in enumerate(english_train_loader, start=1):
    pixel_values = batch["pixel_values"].to(english_model.device)
    labels = batch["labels"].to(english_model.device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        loss = english_model(
            pixel_values=pixel_values,
            labels=labels,
        ).loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(f"Non-finite loss at step {step}")

    # Backpropagate the loss and limit gradient magnitude before updating weights.
    loss.backward()
    torch.nn.utils.clip_grad_norm_(english_model.parameters(), max_norm=1.0)
    optimizer.step()

    total_loss += loss.detach().float().item()

    if step % 16 == 0 or step == total_steps:
        print(
            f"Step {step}/{total_steps} | "
            f"average loss: {total_loss / step:.4f}"
        )

english_model.eval()
average_loss = total_loss / total_steps
del optimizer

# Save the fine-tuned model separately from the pretrained baseline.
english_checkpoint = CHECKPOINT_DIR / "english_trocr_public_epoch1"
english_model.save_pretrained(english_checkpoint)
english_processor.save_pretrained(english_checkpoint)

# Save the run settings and average loss alongside the separately saved model.
training_record = {
    "model": ENGLISH_MODEL_ID,
    "dataset": PUBLIC_DATASETS["English"],
    "training_lines": 512,
    "epochs": 1,
    "batch_size": ENGLISH_BATCH_SIZE,
    "learning_rate": learning_rate,
    "average_training_loss": average_loss,
}
(RESULTS_DIR / "english_training_record.json").write_text(
    json.dumps(training_record, indent=2),
    encoding="utf-8",
)

print(f"\nEpoch complete. Average training loss: {average_loss:.4f}")
print("Checkpoint saved to:", english_checkpoint)

In [ ]:
# Cell 12: Compare English baseline and fine-tuned results

import json
import torch
from jiwer import cer, wer

english_validation = PUBLIC_SUBSETS["English"]["validation"]
references = [text.strip() for text in english_validation["text"]]
predictions = []

english_model.eval()

# Evaluate the updated English weights using the same validation subset and decoding settings.
with torch.inference_mode():
    for start in range(0, len(english_validation), 8):
        batch = english_validation[start : start + 8]
        images = [image.convert("RGB") for image in batch["image"]]

        pixel_values = english_processor(
            images=images, return_tensors="pt"
        ).pixel_values.to(english_model.device)

        token_ids = english_model.generate(
            pixel_values, max_new_tokens=128
        )
        predictions.extend(
            text.strip()
            for text in english_processor.batch_decode(
                token_ids, skip_special_tokens=True
            )
        )

assert len(predictions) == len(references)

# Load the recorded baseline and confirm that the reference lines match exactly.
baseline = json.loads(
    (RESULTS_DIR / "english_baseline_validation.json").read_text(
        encoding="utf-8"
    )
)
assert baseline["references"] == references

finetuned = {
    "model": str(english_checkpoint),
    "dataset": PUBLIC_DATASETS["English"],
    "split": "validation",
    "references": references,
    "predictions": predictions,
    "cer": cer(references, predictions),
    "wer": wer(references, predictions),
}

output_path = RESULTS_DIR / "english_finetuned_validation.json"
output_path.write_text(
    json.dumps(finetuned, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Express changes in percentage points: positive values mean more errors.
cer_change = (finetuned["cer"] - baseline["cer"]) * 100
wer_change = (finetuned["wer"] - baseline["wer"]) * 100

print(f"Character error: {baseline['cer']:.2%} → {finetuned['cer']:.2%}")
print(f"Word error:      {baseline['wer']:.2%} → {finetuned['wer']:.2%}")
print(f"CER change: {cer_change:+.2f} percentage points")
print(f"WER change: {wer_change:+.2f} percentage points")
print("First reference: ", repr(references[0]))
print("First prediction:", repr(predictions[0]))
print("Results saved to:", output_path)

In [ ]:
# Cell 13: Check Chinese vocabulary coverage and CTC alignment

from collections import Counter
import torch

chinese_train = PUBLIC_SUBSETS["Chinese"]["train"]

# In this recogniser, ID 0 is reserved for the CTC blank symbol.
characters = chinese_processor.character_list
chinese_char_to_id = {
    character: index
    for index, character in enumerate(characters)
    if index != 0
}

training_texts = chinese_train["text"]

# Keep only lines whose every character exists in the model vocabulary.
missing_characters = Counter(
    character
    for text in training_texts
    for character in text
    if character not in chinese_char_to_id
)

CHINESE_TRAIN_INDICES = [
    index
    for index, text in enumerate(training_texts)
    if text and all(character in chinese_char_to_id for character in text)
]

print(f"Model vocabulary entries: {len(characters):,}")
print(f"Training lines available: {len(chinese_train):,}")
print(f"Lines with supported labels: {len(CHINESE_TRAIN_INDICES):,}")
print(f"Most common unsupported characters: {missing_characters.most_common(10)}")

if not CHINESE_TRAIN_INDICES:
    raise RuntimeError(
        "No training lines have fully supported labels. "
        "Share this cell's output before proceeding."
    )

# Inspect one supported example without updating model weights.
example = chinese_train[CHINESE_TRAIN_INDICES[0]]
text = example["text"]

chinese_model.eval()
with torch.inference_mode():
    inputs = chinese_processor(
        images=example["image"].convert("RGB"),
        return_tensors="pt",
    ).to(chinese_model.device)
    output = chinese_model(**inputs)

scores = output.last_hidden_state
# Adjacent repeated characters need a separating CTC blank, increasing the required steps.
minimum_ctc_steps = len(text) + sum(
    left == right for left, right in zip(text, text[1:])
)

print(f"Model output shape (batch, time, vocabulary): {tuple(scores.shape)}")
print(f"Scores sum at first time step: {scores[0, 0].sum().item():.4f}")
print(f"Example target length: {len(text)} characters")
print(f"Minimum CTC time steps for this target: {minimum_ctc_steps}")
print(f"Available model time steps: {scores.shape[1]}")
print("Chinese preparation check complete; weights have not changed.")

In [ ]:
# Cell 14: Prepare Chinese batches and verify the CTC loss

import torch
from torch.utils.data import DataLoader, Subset

# Prepare image tensors and character targets in the format expected by CTC loss.
def collate_chinese(examples):
    """Turn four line images and their texts into one training batch."""
    texts = [example["text"] for example in examples]
    images = [example["image"].convert("RGB") for example in examples]

    pixel_values = chinese_processor(
        images=images,
        return_tensors="pt",
    )["pixel_values"]

    # CTC expects all target character IDs joined into one vector.
    target_ids = torch.tensor(
        [chinese_char_to_id[character] for text in texts for character in text],
        dtype=torch.long,
    )
    target_lengths = torch.tensor(
        [len(text) for text in texts],
        dtype=torch.long,
    )

    return pixel_values, target_ids, target_lengths, texts


# Batch supported labels only, with a seeded generator for reproducible shuffling.
chinese_train_loader = DataLoader(
    Subset(chinese_train, CHINESE_TRAIN_INDICES),
    batch_size=4,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
    num_workers=0,
    collate_fn=collate_chinese,
)

pixel_values, target_ids, target_lengths, texts = next(
    iter(chinese_train_loader)
)

# Run one batch without recording gradients or changing weights.
chinese_model.eval()
with torch.no_grad():
    outputs = chinese_model(
        pixel_values=pixel_values.to(chinese_model.device)
    )
    probabilities = outputs.last_hidden_state.float()

    batch_size, time_steps, vocabulary_size = probabilities.shape
    input_lengths = torch.full(
        (batch_size,), time_steps, dtype=torch.long
    )

    # Repeated neighbouring characters need an extra CTC time step.
    minimum_steps = [
        len(text) + sum(a == b for a, b in zip(text, text[1:]))
        for text in texts
    ]

    if max(minimum_steps) <= time_steps:
        # CTC consumes log probabilities in (time, batch, vocabulary) order.
        log_probs = probabilities.clamp_min(1e-30).log()
        log_probs = log_probs.transpose(0, 1).contiguous()

        ctc_loss = torch.nn.CTCLoss(
            blank=0,
            zero_infinity=False,
        )
        batch_loss = ctc_loss(
            log_probs,
            target_ids.to(chinese_model.device),
            input_lengths,
            target_lengths,
        )
        print(f"Initial batch loss: {batch_loss.item():.4f}")
        print(f"Loss is finite: {torch.isfinite(batch_loss).item()}")
    else:
        print("This batch needs more time steps than the model produced.")
        print("We will adjust the batching before training.")

print(f"Image batch shape: {tuple(pixel_values.shape)}")
print(f"Model output shape: {tuple(probabilities.shape)}")
print(f"Target lengths: {target_lengths.tolist()}")
print(f"Minimum required time steps: {minimum_steps}")
print(f"Available time steps: {time_steps}")
print("Chinese training batch checked; weights have not changed.")

In [ ]:
# Cell 15: Fine-tune Chinese recognition with fixed BatchNorm statistics

import json
import time
from pathlib import Path

import torch
from transformers import AutoModelForTextRecognition

# Always begin this training run with the original pretrained weights.
chinese_model = AutoModelForTextRecognition.from_pretrained(
    CHINESE_MODEL_ID,
    dtype=torch.float32,
).to(DEVICE)

# Reset the batch order so this run does not depend on earlier diagnostics.
chinese_train_loader.generator.manual_seed(42)

chinese_model.train()

# Keep pretrained BatchNorm running statistics fixed for these small batches.
# Other trainable parameters, including BatchNorm affine parameters, still receive updates.
bn_types = (
    torch.nn.BatchNorm1d,
    torch.nn.BatchNorm2d,
    torch.nn.BatchNorm3d,
    torch.nn.SyncBatchNorm,
)
frozen_bn_count = 0
for module in chinese_model.modules():
    if isinstance(module, bn_types):
        module.eval()
        frozen_bn_count += 1

# Use a small learning rate for adaptation; blank=0 matches the Chinese vocabulary.
optimizer = torch.optim.AdamW(
    chinese_model.parameters(),
    lr=1e-6,
)
ctc_loss_fn = torch.nn.CTCLoss(
    blank=0,
    zero_infinity=False,
)

completed_steps = 0
skipped_batches = 0
loss_sum = 0.0
total_batches = len(chinese_train_loader)
start_time = time.time()

print(f"BatchNorm layers kept in evaluation mode: {frozen_bn_count}")
print(f"Training on {len(CHINESE_TRAIN_INDICES)} lines for up to {total_batches} steps...")

for batch_number, (pixel_values, target_ids, target_lengths, texts) in enumerate(
    chinese_train_loader, start=1
):
    optimizer.zero_grad(set_to_none=True)

    outputs = chinese_model(
        pixel_values=pixel_values.to(chinese_model.device)
    )
    probabilities = outputs.last_hidden_state.float()
    batch_size, time_steps, _ = probabilities.shape

    # Skip targets that cannot fit the available CTC sequence, including repeated characters.
    minimum_steps = [
        len(text) + sum(a == b for a, b in zip(text, text[1:]))
        for text in texts
    ]
    if max(minimum_steps) > time_steps:
        skipped_batches += 1
        continue

    # Convert character probabilities to the log-probability format required by CTC.
    log_probs = probabilities.clamp_min(1e-30).log()
    log_probs = log_probs.transpose(0, 1).contiguous()
    input_lengths = torch.full(
        (batch_size,), time_steps, dtype=torch.long
    )

    loss = ctc_loss_fn(
        log_probs,
        target_ids.to(chinese_model.device),
        input_lengths,
        target_lengths,
    )

    # Reject invalid losses or gradients so an unstable batch cannot update the model.
    if not torch.isfinite(loss):
        skipped_batches += 1
        continue

    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        chinese_model.parameters(), max_norm=1.0
    )

    if not torch.isfinite(gradient_norm):
        optimizer.zero_grad(set_to_none=True)
        skipped_batches += 1
        continue

    optimizer.step()
    completed_steps += 1
    loss_sum += loss.item()

    if batch_number % 16 == 0 or batch_number == total_batches:
        print(
            f"Batch {batch_number}/{total_batches} | "
            f"average loss: {loss_sum / completed_steps:.4f} | "
            f"skipped: {skipped_batches}"
        )

if completed_steps == 0:
    raise RuntimeError("No valid training steps completed; no checkpoint was saved.")

chinese_model.eval()

# Save the completed model separately from the baseline and record the training outcome.
chinese_checkpoint_frozen_bn = (
    Path(CHECKPOINT_DIR) / "chinese_ppocr_public_frozen_bn_epoch1"
)
chinese_model.save_pretrained(chinese_checkpoint_frozen_bn)
chinese_processor.save_pretrained(chinese_checkpoint_frozen_bn)

training_record = {
    "training_lines": len(CHINESE_TRAIN_INDICES),
    "completed_steps": completed_steps,
    "skipped_batches": skipped_batches,
    "frozen_batchnorm_layers": frozen_bn_count,
    "learning_rate": 1e-6,
    "average_training_loss": loss_sum / completed_steps,
    "elapsed_seconds": round(time.time() - start_time, 1),
    "checkpoint": str(chinese_checkpoint_frozen_bn),
}

record_path = Path(RESULTS_DIR) / "chinese_training_frozen_bn_epoch1.json"
with record_path.open("w", encoding="utf-8") as file:
    json.dump(training_record, file, ensure_ascii=False, indent=2)

print(f"Epoch complete. Average training loss: {loss_sum / completed_steps:.4f}")
print(f"Completed steps: {completed_steps}/{total_batches}")
print(f"Checkpoint saved to: {chinese_checkpoint_frozen_bn}")
print(f"Training record saved to: {record_path}")

In [ ]:
# Cell 16: Evaluate the fine-tuned Chinese recogniser

import json
import unicodedata
from pathlib import Path
from jiwer import cer

import torch

validation_data = PUBLIC_SUBSETS["Chinese"]["validation"]
frozen_bn_references = []
frozen_bn_predictions = []

# Evaluate the fine-tuned model with fixed inference behaviour and no gradient tracking.
chinese_model.eval()
with torch.inference_mode():
    for start in range(0, len(validation_data), 4):
        examples = validation_data[
            start : min(start + 4, len(validation_data))
        ]

        images = [image.convert("RGB") for image in examples["image"]]
        inputs = chinese_processor(
            images=images,
            return_tensors="pt",
        ).to(chinese_model.device)

        outputs = chinese_model(**inputs)
        decoded = chinese_processor.post_process_text_recognition(outputs)

        frozen_bn_references.extend(examples["text"])
        frozen_bn_predictions.extend(
            result["text"] for result in decoded
        )

assert (
    len(frozen_bn_references)
    == len(frozen_bn_predictions)
    == len(validation_data)
)

# Report both exact-character CER and CER after Unicode NFKC normalization.
frozen_bn_raw_cer = cer(
    frozen_bn_references,
    frozen_bn_predictions,
)
frozen_bn_nfkc_cer = cer(
    [
        unicodedata.normalize("NFKC", text)
        for text in frozen_bn_references
    ],
    [
        unicodedata.normalize("NFKC", text)
        for text in frozen_bn_predictions
    ],
)

# Save every reference/prediction pair with the scores so the result can be inspected.
result_path = (
    Path(RESULTS_DIR) / "chinese_finetuned_frozen_bn_validation.json"
)
with result_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "split": "validation",
            "number_of_lines": len(frozen_bn_references),
            "exact_character_error_rate": frozen_bn_raw_cer,
            "nfkc_character_error_rate": frozen_bn_nfkc_cer,
            "references": frozen_bn_references,
            "predictions": frozen_bn_predictions,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

# Display the saved baseline beside this run; lower CER indicates fewer character errors.
baseline_path = Path(RESULTS_DIR) / "chinese_baseline_validation.json"
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))

print(f"Baseline exact CER:       {baseline['cer']:.2%}")
print(f"Revised model exact CER:  {frozen_bn_raw_cer:.2%}")
print(f"Baseline NFKC CER:        {baseline['cer_nfkc']:.2%}")
print(f"Revised model NFKC CER:   {frozen_bn_nfkc_cer:.2%}")

In [ ]:
# Cell 17: Model loading, inference, and shared metrics
import gc, json, time, unicodedata
from pathlib import Path
from contextlib import nullcontext

import torch
import pandas as pd
from huggingface_hub import hf_hub_download
from transformers import (
    AutoImageProcessor,
    AutoModelForTextRecognition,
    RobertaTokenizer,
    TrOCRProcessor,
    ViTImageProcessor,
    VisionEncoderDecoderModel,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE = {
    'en': 'microsoft/trocr-base-handwritten',
    'zh': 'PaddlePaddle/PP-OCRv5_server_rec_safetensors',
}
PUBLIC = {
    'en': Path(CHECKPOINT_DIR) / 'english_trocr_public_epoch1',
    'zh': Path(CHECKPOINT_DIR) / 'chinese_ppocr_public_frozen_bn_epoch1',
}

RUN_ID = 'v1'  # Use a NEW ID when changing data or training settings.
ADAPTED = {
    lang: Path(CHECKPOINT_DIR) / 'curated' / f'{lang}_{RUN_ID}'
    for lang in BASE
}
MAX_NEW_TOKENS = 128

# Release model/tensor leftovers from the public-data cells.
for _name, _value in list(globals().items()):
    if isinstance(
        _value, (torch.nn.Module, torch.optim.Optimizer, torch.Tensor)
    ):
        globals().pop(_name, None)

for _name in ('outputs', 'output', 'check_batch', 'inputs', '_OCR'):
    globals().pop(_name, None)

del _value
_OCR = None


def release_ocr():
    global _OCR
    if _OCR is not None:
        _OCR[1].cpu()
    _OCR = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


release_ocr()


def has_weights(path):
    return (path / 'config.json').is_file() and (
        any(path.glob('*.safetensors'))
        or any(path.glob('pytorch_model*.bin'))
    )


def model_sources(lang):
    sources = {'baseline': BASE[lang]}

    if has_weights(PUBLIC[lang]):
        sources['public'] = str(PUBLIC[lang])

    marker = ADAPTED[lang] / 'selected.json'
    if marker.is_file():
        selected = read_json(marker)
        path = ADAPTED[lang] / selected['folder']
        if has_weights(path):
            sources['adapted'] = str(path)

    return sources


def load_ocr(lang, version):
    global _OCR
    source = model_sources(lang).get(version)
    if source is None:
        raise FileNotFoundError(
            f'{lang}/{version}: no saved checkpoint is available.'
        )

    if _OCR is not None and _OCR[0] == (lang, source):
        return _OCR[1], _OCR[2]

    release_ocr()

    if lang == 'en':
        def tokenizer_file(name):
            local = Path(source) / name
            return (
                str(local) if local.is_file()
                else hf_hub_download(BASE[lang], name)
            )

        tokenizer = RobertaTokenizer(
            vocab=tokenizer_file('vocab.json'),
            merges=tokenizer_file('merges.txt'),
            model_max_length=512,
        )
        processor = TrOCRProcessor(
            ViTImageProcessor.from_pretrained(source), tokenizer
        )
        model = VisionEncoderDecoderModel.from_pretrained(
            source, dtype=torch.float32
        )
        model.config.decoder_start_token_id = tokenizer.eos_token_id
        model.config.pad_token_id = tokenizer.pad_token_id
        model.config.vocab_size = model.config.decoder.vocab_size
    else:
        processor = AutoImageProcessor.from_pretrained(source)
        model = AutoModelForTextRecognition.from_pretrained(
            source, dtype=torch.float32
        )

    model.to(DEVICE).eval()
    _OCR = ((lang, source), model, processor)
    return model, processor


def open_line(row):
    with Image.open(row['local_path']) as image:
        return image.convert('RGB')


@torch.inference_mode()
def transcribe(image, lang, model, processor):
    model.eval()
    inputs = processor(
        images=image.convert('RGB'), return_tensors='pt'
    ).to(DEVICE)

    if lang == 'en':
        tokens = model.generate(
            inputs['pixel_values'],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            decoder_start_token_id=processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
        return processor.batch_decode(
            tokens, skip_special_tokens=True
        )[0].strip()

    output = model(**inputs)
    return processor.post_process_text_recognition(
        output
    )[0]['text'].strip()


def distance(a, b):
    previous = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        current = [i]
        for j, y in enumerate(b, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (x != y),
            ))
        previous = current
    return previous[-1]


def error_rates(references, predictions, lang):
    def rate(refs, preds):
        return (
            sum(distance(r, p) for r, p in zip(refs, preds))
            / sum(map(len, refs))
        )

    return {
        # Corpus-level CER; preserves case, punctuation and spacing.
        'CER': rate(references, predictions),
        'CER_NFKC': rate(
            [unicodedata.normalize('NFKC', x) for x in references],
            [unicodedata.normalize('NFKC', x) for x in predictions],
        ),
        'WER': (
            rate(
                [x.split() for x in references],
                [x.split() for x in predictions],
            )
            if lang == 'en' else None
        ),
    }


def curated_rows(split, lang):
    if not CURATED_READY:
        raise RuntimeError('Run Cell 4A successfully first.')

    return sorted(
        (
            r for r in CURATED_ROWS
            if r['split'] == split and r['language'] == lang
        ),
        key=lambda r: r['sample_id'],
    )


print(
    'Available checkpoints:',
    {lang: list(model_sources(lang)) for lang in BASE},
)

In [ ]:
# Cell 18: Additional adaptation from the PUBLIC-data checkpoint
LANGUAGE = 'en'  # 'en' or 'zh'; one recogniser at a time.
EPOCHS = 1
BATCH_SIZE = 4
SAVE_EVERY = 50
SEED = 42


def adapt_curated(lang):
    train = curated_rows('train', lang)
    validation = curated_rows('validation', lang)

    if not train:
        print(f'{lang}: no curated training samples; adaptation skipped.')
        return

    if 'public' not in model_sources(lang):
        print(
            'Run public training first. Required checkpoint:',
            PUBLIC[lang],
        )
        return

    if EPOCHS < 1 or BATCH_SIZE < 1 or SAVE_EVERY < 1:
        raise ValueError(
            'EPOCHS, BATCH_SIZE and SAVE_EVERY must be positive.'
        )

    if not validation:
        print(
            'No validation samples: fixed epochs only; '
            'saved model is UNVALIDATED.'
        )

    lr = 5e-6 if lang == 'en' else 1e-6
    bf16 = (
        lang == 'en'
        and DEVICE == 'cuda'
        and torch.cuda.is_bf16_supported()
    )

    settings = dict(
        train=fingerprint(train),
        validation=fingerprint(validation),
        start=str(PUBLIC[lang]),
        learning_rate=lr,
        batch_size=BATCH_SIZE,
        seed=SEED,
        bf16=bf16,
        max_tokens=MAX_NEW_TOKENS,
        start_files=[
            (p.name, p.stat().st_size, p.stat().st_mtime_ns)
            for p in sorted(PUBLIC[lang].iterdir())
            if p.is_file()
        ],
    )
    settings = json.loads(json.dumps(settings))

    folder = ADAPTED[lang]
    folder.mkdir(parents=True, exist_ok=True)
    config_file = folder / 'run.json'
    resume_file = folder / 'resume.pt'

    if config_file.exists() and read_json(config_file) != settings:
        raise ValueError(
            'Data/start checkpoint/settings changed. '
            'Use a new RUN_ID in Cell 17.'
        )

    config_file.write_text(
        json.dumps(settings, indent=2), encoding='utf-8'
    )
    print('Starting weights:', PUBLIC[lang])
    print('Training lines:', len(train), '| Adapted output:', folder)

    release_ocr()
    torch.manual_seed(SEED)
    model, processor = load_ocr(lang, 'public')

    # Do not cache changing weights under the "public" identity.
    global _OCR
    _OCR = ((lang, 'training-in-progress'), model, processor)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    state = dict(epoch=0, offset=0, best=float('inf'), history=[])

    def save_resume():
        payload = dict(
            state,
            settings=settings,
            model=model.state_dict(),
            optimizer=optimizer.state_dict(),
            rng=torch.get_rng_state(),
            cuda_rng=(
                torch.cuda.get_rng_state_all()
                if DEVICE == 'cuda' else []
            ),
        )
        temporary = folder / 'resume.tmp'
        torch.save(payload, temporary)
        temporary.replace(resume_file)

    try:
        if resume_file.exists():
            saved = torch.load(
                resume_file, map_location='cpu', weights_only=True
            )
            if saved['settings'] != settings:
                raise ValueError(
                    'Resume checkpoint does not match this run.'
                )

            model.load_state_dict(saved['model'])
            optimizer.load_state_dict(saved['optimizer'])
            state = {k: saved[k] for k in state}
            torch.set_rng_state(saved['rng'])

            if DEVICE == 'cuda':
                torch.cuda.set_rng_state_all(saved['cuda_rng'])

            del saved
            print(
                'Resuming: completed epochs =', state['epoch'],
                '| next offset =', state['offset'],
            )

        if lang == 'en':
            tokenizer = processor.tokenizer
            too_long = [
                r['sample_id'] for r in train
                if len(tokenizer(r['text']).input_ids) - 1
                > MAX_NEW_TOKENS
            ]
            if too_long:
                raise ValueError(
                    'Labels exceed token budget; no labels were truncated: '
                    f'{too_long[:10]}'
                )
        else:
            char_ids = {
                c: i for i, c in enumerate(processor.character_list)
                if i != 0
            }
            unsupported = sorted(
                set(''.join(r['text'] for r in train)) - set(char_ids)
            )
            if unsupported:
                raise ValueError(
                    'Chinese labels contain unsupported characters: '
                    f'{unsupported!r}'
                )
            ctc = torch.nn.CTCLoss(blank=0, zero_infinity=False)

        while state['epoch'] < EPOCHS:
            epoch = state['epoch']
            order = torch.randperm(
                len(train),
                generator=torch.Generator().manual_seed(SEED + epoch),
            ).tolist()

            model.train()
            if lang == 'zh':
                for module in model.modules():
                    if isinstance(
                        module, torch.nn.modules.batchnorm._BatchNorm
                    ):
                        module.eval()

            for offset in range(
                state['offset'], len(train), BATCH_SIZE
            ):
                batch = [
                    train[i]
                    for i in order[offset:offset + BATCH_SIZE]
                ]
                texts = [r['text'] for r in batch]
                pixels = processor(
                    images=[open_line(r) for r in batch],
                    return_tensors='pt',
                )['pixel_values'].to(DEVICE)

                optimizer.zero_grad(set_to_none=True)

                if lang == 'en':
                    tokens = tokenizer(
                        texts, padding=True, return_tensors='pt'
                    )
                    labels = tokens.input_ids[:, 1:].clone()
                    labels[tokens.attention_mask[:, 1:] == 0] = -100

                    context = (
                        torch.autocast('cuda', dtype=torch.bfloat16)
                        if bf16 else nullcontext()
                    )
                    with context:
                        loss = model(
                            pixel_values=pixels,
                            labels=labels.to(DEVICE),
                        ).loss
                else:
                    probabilities = model(
                        pixel_values=pixels
                    ).last_hidden_state.float()
                    steps = probabilities.shape[1]

                    required_steps = [
                        len(t) + sum(
                            a == b for a, b in zip(t, t[1:])
                        )
                        for t in texts
                    ]
                    if max(required_steps) > steps:
                        raise ValueError(
                            'A Chinese label cannot fit the CTC output. '
                            'No batch was discarded.'
                        )

                    if not torch.allclose(
                        probabilities.sum(-1),
                        torch.ones_like(probabilities[..., 0]),
                        atol=1e-3,
                    ):
                        raise ValueError(
                            'Expected PP-OCR probability output; '
                            'inspect the installed model.'
                        )

                    targets = torch.tensor(
                        [char_ids[c] for t in texts for c in t],
                        dtype=torch.long,
                        device=DEVICE,
                    )
                    loss = ctc(
                        probabilities.clamp_min(1e-30).log()
                        .transpose(0, 1).contiguous(),
                        targets,
                        torch.full(
                            (len(batch),), steps, dtype=torch.long
                        ),
                        torch.tensor(
                            [len(t) for t in texts], dtype=torch.long
                        ),
                    )

                if not torch.isfinite(loss):
                    raise RuntimeError(
                        'Non-finite loss; model update stopped.'
                    )

                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )
                if not torch.isfinite(norm):
                    raise RuntimeError(
                        'Non-finite gradients; model update stopped.'
                    )

                optimizer.step()
                state['offset'] = min(
                    offset + BATCH_SIZE, len(train)
                )

                if (offset // BATCH_SIZE + 1) % SAVE_EVERY == 0:
                    save_resume()
                    print(
                        f'Epoch {epoch + 1}: '
                        f'{state["offset"]}/{len(train)} lines; '
                        f'training loss {loss.item():.4f}'
                    )

            model.eval()
            metrics = None
            if validation:
                predictions = [
                    transcribe(open_line(r), lang, model, processor)
                    for r in validation
                ]
                metrics = error_rates(
                    [r['text'] for r in validation],
                    predictions,
                    lang,
                )
                print(
                    'Validation:', metrics,
                    '| lines:', len(validation),
                )

            selected = (
                not validation or metrics['CER'] < state['best']
            )
            if selected:
                checkpoint = folder / f'epoch_{epoch + 1:03d}'
                model.save_pretrained(checkpoint)
                processor.save_pretrained(checkpoint)

                marker = dict(
                    folder=checkpoint.name,
                    epoch=epoch + 1,
                    selection=(
                        'validation_CER'
                        if validation else 'fixed_epochs_unvalidated'
                    ),
                    validation_metrics=metrics,
                    train_fingerprint=fingerprint(train),
                    training_sample_ids=[
                        r['sample_id'] for r in train
                    ],
                    training_writers=sorted({
                        r['writer_id'] for r in train
                    }),
                    training_pages=sorted({
                        r['source_page_id'] for r in train
                    }),
                    training_image_hashes=[
                        r['sha256'] for r in train
                    ],
                    training_texts=[r['text'] for r in train],
                )
                temporary = folder / 'selected.tmp'
                temporary.write_text(
                    json.dumps(
                        marker, ensure_ascii=False, indent=2
                    ),
                    encoding='utf-8',
                )
                temporary.replace(folder / 'selected.json')

                if validation:
                    state['best'] = metrics['CER']

            state['history'].append(
                dict(epoch=epoch + 1, validation=metrics)
            )
            state.update(epoch=epoch + 1, offset=0)
            save_resume()

        print(
            'Completed. Selected model:',
            read_json(folder / 'selected.json')['folder'],
        )

    finally:
        del optimizer
        release_ocr()


adapt_curated(LANGUAGE)

In [ ]:
# Cell 19: Fair held-out comparison
EVALUATION_SPLIT = 'validation'
RUN_FINAL_TEST = False


def compare_curated(split):
    if split not in ('validation', 'test'):
        raise ValueError(
            'Only validation or test evaluation is allowed.'
        )

    if split == 'test' and not RUN_FINAL_TEST:
        print(
            'Test reserved. Freeze model/settings, '
            'then set RUN_FINAL_TEST=True.'
        )
        return

    summaries, predictions = [], []

    for lang in ('en', 'zh'):
        rows = curated_rows(split, lang)
        if not rows:
            print(
                f'{split}/{lang}: 0 lines; '
                'held-out metrics unavailable.'
            )
            continue

        sources = model_sources(lang)
        for version in ('baseline', 'public', 'adapted'):
            if version not in sources:
                print(
                    f'{lang}/{version}: checkpoint unavailable; '
                    'comparison is incomplete.'
                )

        training = None
        if 'adapted' in sources:
            training = read_json(
                ADAPTED[lang] / 'selected.json'
            )
            for field, saved_field in (
                ('sample_id', 'training_sample_ids'),
                ('writer_id', 'training_writers'),
                ('source_page_id', 'training_pages'),
                ('sha256', 'training_image_hashes'),
            ):
                if (
                    {r[field] for r in rows}
                    & set(training[saved_field])
                ):
                    raise ValueError(
                        'Held-out data overlaps saved adaptation '
                        f'training: {field}'
                    )

        seen_texts = (
            set(training['training_texts']) if training else set()
        )
        print(
            f'{split}/{lang}: {len(rows)} identical lines '
            'per available model; writers:',
            sorted({
                (r['writer_id'], r['writer_name']) for r in rows
            }),
        )

        try:
            for version, source in sources.items():
                model, processor = load_ocr(lang, version)
                texts = [
                    transcribe(open_line(r), lang, model, processor)
                    for r in rows
                ]

                for r, text in zip(rows, texts):
                    predictions.append(dict(
                        split=split,
                        language=lang,
                        model=version,
                        source=source,
                        sample_id=r['sample_id'],
                        writer_id=r['writer_id'],
                        writer_name=r['writer_name'],
                        source_page_id=r['source_page_id'],
                        reference=r['text'],
                        prediction=text,
                    ))

                groups = ['ALL'] + sorted({
                    r['writer_id'] for r in rows
                })
                for writer in groups:
                    indices = [
                        i for i, r in enumerate(rows)
                        if writer == 'ALL'
                        or r['writer_id'] == writer
                    ]
                    refs = [rows[i]['text'] for i in indices]
                    preds = [texts[i] for i in indices]

                    summaries.append(dict(
                        split=split,
                        language=lang,
                        model=version,
                        writer=writer,
                        lines=len(indices),
                        pages=len({
                            rows[i]['source_page_id']
                            for i in indices
                        }),
                        reference_characters=sum(map(len, refs)),
                        exact_text_seen_in_adaptation=(
                            sum(t in seen_texts for t in refs)
                            if training else None
                        ),
                        **error_rates(refs, preds, lang),
                    ))
        finally:
            release_ocr()

    if summaries:
        display(pd.DataFrame(summaries))

        destination = Path(RESULTS_DIR) / (
            f'curated_{split}_{time.time_ns()}.json'
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(
            json.dumps(
                dict(
                    split=split,
                    metrics=summaries,
                    predictions=predictions,
                    notes=(
                        'CER uses Unicode characters; WER uses English '
                        'whitespace-delimited words. NFKC CER is additional. '
                        'Exact-text overlap refers only to curated '
                        'adaptation labels.'
                    ),
                    decoding=dict(
                        max_new_tokens=MAX_NEW_TOKENS,
                        num_beams=1,
                        do_sample=False,
                    ),
                ),
                ensure_ascii=False,
                indent=2,
            ),
            encoding='utf-8',
        )
        print(
            'Full references, predictions and coverage saved:',
            destination,
        )


compare_curated(EVALUATION_SPLIT)

In [ ]:
# Cell 19A: Final test evaluation (public AND curated)
# Keep False while choosing hyperparameters/checkpoints on validation.
# Set True once the settings/checkpoints are frozen, then run this cell.
RUN_FINAL_TEST = False
PUBLIC_TEST_LIMIT = None  # None = full official test split; integer = labeled subset only.

def evaluate_public_test(limit=None):
    from datasets import load_dataset
    if limit is not None and (not isinstance(limit, int) or limit < 1):
        raise ValueError('PUBLIC_TEST_LIMIT must be None or a positive integer')
    for language, lang in (('English', 'en'), ('Chinese', 'zh')):
        # Load test only here; never use test examples in training or selection.
        dataset = load_dataset(PUBLIC_DATASETS[language], split='test', streaming=True)
        if limit is not None:
            dataset = dataset.take(limit)
        # Cache one split locally so all models see identical lines.
        rows = list(dataset)
        sources = model_sources(lang)
        print(language, 'test lines:', len(rows), '| versions:', list(sources))
        if 'public' not in sources:
            raise FileNotFoundError(f'{lang}: public fine-tuned checkpoint required')
        try:
            for version, source in sources.items():
                model, processor = load_ocr(lang, version)
                result = evaluate_rows(rows, lambda row: transcribe(
                    row['image'], lang, model, processor), lang)
                result.update(dataset=PUBLIC_DATASETS[language], split='test',
                              language=lang, model=version, source=source,
                              coverage='full' if limit is None else 'subset',
                              requested_limit=limit,
                              decoding=dict(max_new_tokens=MAX_NEW_TOKENS,
                                            num_beams=1, do_sample=False))
                destination = Path(RESULTS_DIR) / f'{lang}_{version}_test_{time.time_ns()}.json'
                destination.write_text(json.dumps(result, ensure_ascii=False, indent=2),
                                       encoding='utf-8')
                print(version, {k: result[k] for k in ('CER', 'CER_NFKC', 'CER_PUNCT', 'WER')})
                print('Saved:', destination)
        finally:
            release_ocr()

if RUN_FINAL_TEST:
    evaluate_public_test(PUBLIC_TEST_LIMIT)
    if globals().get('CURATED_READY', False):
        compare_curated('test')
    else:
        print('Curated test unavailable: Cell 4A has not completed successfully.')
else:
    print('Final test ready. Freeze settings, set RUN_FINAL_TEST=True, then run this cell.')


In [ ]:
# Cell 19B: Configure and load the translation baseline
import gc
import json
import time
from pathlib import Path

import torch
import transformers
from transformers import MarianMTModel, MarianTokenizer

MT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# The key identifies the INPUT language; the target is the other language.
MT_MODELS = {
    'en': 'Helsinki-NLP/opus-mt-en-zh',
    'zh': 'Helsinki-NLP/opus-mt-zh-en',
}
MT_ZH_TAG = '>>cmn_Hans<<'  # Request Simplified Chinese output.
MT_MAX_INPUT_TOKENS = 256
MT_MAX_NEW_TOKENS = 128
MT_NUM_BEAMS = 4

# Release a previous translation model if this setup cell is rerun.
if callable(globals().get('release_mt')):
    release_mt()
_MT = None


def release_mt():
    global _MT
    if _MT is not None:
        _MT['model'].cpu()
    _MT = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_mt(source_language):
    global _MT

    if source_language not in MT_MODELS:
        raise ValueError("source_language must be 'en' or 'zh'.")

    # Reuse Cell 17's memory cleanup when OCR has been running.
    if callable(globals().get('release_ocr')):
        release_ocr()

    if _MT is not None and _MT['source_language'] == source_language:
        return _MT

    release_mt()
    model_id = MT_MODELS[source_language]
    started = time.perf_counter()
    print('Loading translation model:', model_id)

    tokenizer = MarianTokenizer.from_pretrained(model_id)
    if source_language == 'en' and MT_ZH_TAG not in tokenizer.get_vocab():
        raise RuntimeError(
            'The tokenizer is missing the Simplified Chinese tag.'
        )

    # Keep FP32 for this initial baseline; optimisation comes later.
    model = MarianMTModel.from_pretrained(
        model_id, dtype=torch.float32
    )
    model.to(MT_DEVICE).eval()

    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()

    _MT = dict(
        source_language=source_language,
        model_id=model_id,
        revision=getattr(model.config, '_commit_hash', None),
        model=model,
        tokenizer=tokenizer,
    )

    print(
        f'Ready on {MT_DEVICE}; '
        f'loading took {time.perf_counter() - started:.2f}s'
    )
    return _MT


print('Translation setup ready. No model training is required.')

In [ ]:
# Cell 19C: Translate a sentence in the selected direction
def translate_text(text, source_language):
    if source_language not in MT_MODELS:
        raise ValueError(
            "Choose 'en' for English input or 'zh' for Chinese input."
        )
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Enter a non-empty sentence.')

    text = text.strip()
    loaded = load_mt(source_language)
    model, tokenizer = loaded['model'], loaded['tokenizer']

    prepared = (
        f'{MT_ZH_TAG} {text}'
        if source_language == 'en'
        else text
    )

    # Time text processing and inference, excluding model loading/downloads.
    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()

    inputs = tokenizer(
        prepared, return_tensors='pt', truncation=False
    )
    input_tokens = inputs['input_ids'].shape[1]
    limit = min(
        MT_MAX_INPUT_TOKENS,
        model.config.max_position_embeddings,
    )
    if input_tokens > limit:
        raise ValueError(
            f'Input has {input_tokens} tokens; limit is {limit}. '
            'Use a shorter sentence.'
        )

    # Fixed decoding settings make later comparisons easier to reproduce.
    with torch.inference_mode():
        generated = model.generate(
            **inputs.to(MT_DEVICE),
            max_length=None,
            max_new_tokens=MT_MAX_NEW_TOKENS,
            num_beams=MT_NUM_BEAMS,
            do_sample=False,
            early_stopping=True,
        )

    translation = tokenizer.decode(
        generated[0], skip_special_tokens=True
    ).strip()

    if MT_DEVICE == 'cuda':
        torch.cuda.synchronize()
    seconds = time.perf_counter() - started

    return dict(
        source_language=source_language,
        target_language=(
            'zh-Hans' if source_language == 'en' else 'en'
        ),
        source_text=text,
        translation=translation,
        model_id=loaded['model_id'],
        revision=loaded['revision'],
        input_tokens=input_tokens,
        translation_seconds=round(seconds, 4),
        # Reaching the cap can mean incomplete output.
        output_limit_reached=(
            generated.shape[1] - 1 >= MT_MAX_NEW_TOKENS
        ),
    )


print("Ready: translate_text('Your sentence', 'en' or 'zh').")

In [ ]:
# Cell 19D: Check both directions using typed sentences

# These examples are functional checks, not a scored evaluation dataset.
MT_EXAMPLES = [
    ('en', 'Computer vision is an interesting subject.'),
    ('en', 'Please close the door.'),
    ('zh', '计算机视觉是一门有趣的学科。'),
    ('zh', '请关门。'),
]

mt_results = []

try:
    for source_language, sentence in MT_EXAMPLES:
        result = translate_text(sentence, source_language)
        mt_results.append(result)

        print(
            f"\n{result['source_language']} "
            f"-> {result['target_language']}"
        )
        print('Input: ', result['source_text'])
        print('Output:', result['translation'])
        print(
            f"Translation time: {result['translation_seconds']:.3f}s "
            '(loading excluded)'
        )

        if result['output_limit_reached']:
            print(
                'Output reached the token limit; '
                'it may be incomplete.'
            )
finally:
    # Leave GPU memory available for the existing final OCR interface.
    release_mt()

# Save outputs and settings so this initial run can be reviewed later.
record = dict(
    purpose=(
        'Part C step 1: functional translation checks, '
        'not benchmark scores'
    ),
    models=MT_MODELS,
    device=MT_DEVICE,
    hardware=(
        torch.cuda.get_device_name(0)
        if MT_DEVICE == 'cuda' else 'CPU'
    ),
    transformers_version=transformers.__version__,
    torch_version=str(torch.__version__),
    decoding=dict(
        num_beams=MT_NUM_BEAMS,
        max_new_tokens=MT_MAX_NEW_TOKENS,
        do_sample=False,
    ),
    results=mt_results,
)

result_path = Path(RESULTS_DIR) / f'mt_step1_{time.time_ns()}.json'
result_path.parent.mkdir(parents=True, exist_ok=True)
result_path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('\nSaved:', result_path)

In [ ]:
# Cell 19E: Load a fixed FLORES+ development subset for Part C
import hashlib
import json
import random
from getpass import getpass
from pathlib import Path

from huggingface_hub import get_token, hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

MT_DATA_READY = False
MT_DEV = {}
MT_DATASET = 'openlanguagedata/flores_plus'
MT_DATA_REVISION = '1147e2dc59f37fbb5c22fd8478770de02cd399f9'
MT_DEV_SIZE = 200
MT_DATA_SEED = 42

if 'PROJECT_DIR' not in globals() or not Path(PROJECT_DIR).is_dir():
    raise RuntimeError('Run Cell 4 to mount Drive and prepare the project folder.')

MT_DATA_DIR = Path(PROJECT_DIR) / 'data' / 'translation' / 'flores_plus'
MT_DATA_DIR.mkdir(parents=True, exist_ok=True)
MT_DEV_CACHE = MT_DATA_DIR / (
    f'dev_{MT_DEV_SIZE}_seed{MT_DATA_SEED}_{MT_DATA_REVISION[:12]}.json'
)


def mt_data_hash(pairs):
    raw = json.dumps(pairs, ensure_ascii=False, sort_keys=True).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()


def mt_read_reference_file(path, language, script):
    # Match by ID, never by assuming the two files have the same row order.
    rows = {}
    for line in Path(path).read_text(encoding='utf-8-sig').splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        sample_id = str(row['id'])
        text = row['text']

        if row.get('split', 'dev') != 'dev':
            raise ValueError('Unexpected split: only dev is allowed in this cell.')
        if (row['iso_639_3'], row['iso_15924']) != (language, script):
            raise ValueError(f'Unexpected language/script in {Path(path).name}.')
        if row['id'] is None or not sample_id.strip() or sample_id in rows:
            raise ValueError(f'Missing or duplicate sentence ID: {sample_id!r}')
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f'Empty reference text for sentence {sample_id}.')

        rows[sample_id] = text
    return rows


def mt_prepare_dev():
    settings = dict(
        dataset=MT_DATASET, revision=MT_DATA_REVISION, split='dev',
        languages={'en': 'eng_Latn', 'zh': 'cmn_Hans'},
        subset_size=MT_DEV_SIZE, seed=MT_DATA_SEED,
    )

    if MT_DEV_CACHE.exists():
        payload = json.loads(MT_DEV_CACHE.read_text(encoding='utf-8'))
        if payload['settings'] != settings:
            raise ValueError('Saved dataset settings differ; inspect the cache.')
        print('Reusing the saved development subset.')
    else:
        # Use an existing HF login/Colab secret, or a hidden token prompt.
        # The token stays inside this function and is never saved to Drive.
        token = get_token() or getpass('Hugging Face read token (hidden): ').strip()
        if not token:
            raise ValueError('A token from an account with FLORES+ access is required.')

        try:
            english_path = hf_hub_download(
                MT_DATASET, 'dev/eng_Latn.jsonl', repo_type='dataset',
                revision=MT_DATA_REVISION, token=token,
            )
            chinese_path = hf_hub_download(
                MT_DATASET, 'dev/cmn_Hans.jsonl', repo_type='dataset',
                revision=MT_DATA_REVISION, token=token,
            )
        except HfHubHTTPError as exc:
            if getattr(exc.response, 'status_code', None) in (401, 403):
                raise RuntimeError(
                    'FLORES+ access was denied. Accept its access conditions on '
                    'Hugging Face, then use a read token from that same account. '
                    'If using the HF_TOKEN Colab secret, check its permissions.'
                ) from None
            raise
        finally:
            del token

        english = mt_read_reference_file(english_path, 'eng', 'Latn')
        chinese = mt_read_reference_file(chinese_path, 'cmn', 'Hans')

        if set(english) != set(chinese):
            raise ValueError('English and Chinese sentence IDs do not match.')
        if len(english) < MT_DEV_SIZE:
            raise ValueError(
                f'Only {len(english)} pairs exist; requested {MT_DEV_SIZE}.'
            )

        # Choose once, independently of model predictions or translation quality.
        selected_ids = random.Random(MT_DATA_SEED).sample(
            sorted(english), MT_DEV_SIZE
        )
        pairs = [
            dict(id=i, en=english[i], zh=chinese[i])
            for i in selected_ids
        ]
        payload = dict(
            settings=settings,
            available_dev_pairs=len(english),
            purpose='Development evaluation only; not model training',
            license='CC-BY-SA-4.0',
            pairs=pairs,
            sha256=mt_data_hash(pairs),
        )

        # Save the selected references so later runs reuse the same examples.
        temporary = MT_DEV_CACHE.with_suffix('.tmp')
        temporary.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        temporary.replace(MT_DEV_CACHE)

    pairs = payload['pairs']
    if mt_data_hash(pairs) != payload['sha256']:
        raise ValueError('Saved reference data changed; inspect the cache.')
    if len(pairs) != MT_DEV_SIZE or len({r['id'] for r in pairs}) != len(pairs):
        raise ValueError('Unexpected subset size or duplicate sentence IDs.')

    return payload


MT_DATA_RECORD = mt_prepare_dev()

# The same sentence pairs support evaluation in both translation directions.
MT_DEV = {
    source: [
        dict(id=row['id'], source=row[source], reference=row[target])
        for row in MT_DATA_RECORD['pairs']
    ]
    for source, target in (('en', 'zh'), ('zh', 'en'))
}
MT_DATA_READY = True

print('Available aligned dev pairs:', MT_DATA_RECORD['available_dev_pairs'])
print('Selected pairs:', len(MT_DATA_RECORD['pairs']))
print('English -> Chinese cases:', len(MT_DEV['en']))
print('Chinese -> English cases:', len(MT_DEV['zh']))
print('Reference fingerprint:', MT_DATA_RECORD['sha256'][:16])
print('Cache:', MT_DEV_CACHE)
print('devtest was not downloaded or evaluated. No model training was performed.')

In [ ]:
# Cell 19F: Set up translation metrics and a reusable timed decoder
import sys
import subprocess

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', 'sacrebleu==2.6.0'
])

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
import torch
import transformers
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF

MT_BENCH_BEAMS = (1, 4)
MT_BENCH_REPEATS = 3
MT_BENCH_WARMUPS = 3

# Allow longer sentences than the four examples in Cell 19D.
MT_BENCH_INPUT_LIMIT = 512
MT_BENCH_OUTPUT_LIMIT = 256


@torch.inference_mode()
def mt_decode_loaded(text, source_language, loaded, beams):
    """Translate with an already-loaded model; never read reference answers."""
    if source_language not in ('en', 'zh') or beams not in MT_BENCH_BEAMS:
        raise ValueError('Choose en/zh and one of MT_BENCH_BEAMS.')
    if loaded['source_language'] != source_language:
        raise ValueError('The loaded model has the wrong translation direction.')
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Source text must be a non-empty string.')

    model, tokenizer = loaded['model'], loaded['tokenizer']
    device = next(model.parameters()).device

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    started = time.perf_counter()

    prepared = text.strip()
    if source_language == 'en':
        prepared = f'{MT_ZH_TAG} {prepared}'

    inputs = tokenizer(prepared, return_tensors='pt', truncation=False)
    input_tokens = inputs['input_ids'].shape[1]
    limit = min(MT_BENCH_INPUT_LIMIT, model.config.max_position_embeddings)

    if input_tokens > limit:
        raise ValueError(f'Input has {input_tokens} tokens; limit is {limit}.')

    generated = model.generate(
        **inputs.to(device),
        max_length=None,
        max_new_tokens=MT_BENCH_OUTPUT_LIMIT,
        num_beams=beams,
        num_return_sequences=1,
        do_sample=False,
        early_stopping=(beams > 1),
        length_penalty=1.0,
        use_cache=True,
    )

    token_ids = generated[0].tolist()
    prediction = tokenizer.decode(
        token_ids, skip_special_tokens=True
    ).strip()

    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - started

    return dict(
        prediction=prediction,
        seconds=elapsed,
        input_tokens=input_tokens,
        output_tokens=len(token_ids) - 1,  # Exclude decoder-start token.
        hit_output_limit=(len(token_ids) - 1 >= MT_BENCH_OUTPUT_LIMIT),
    )


print('Ready: BLEU, chrF and timed decoding. No benchmark has run yet.')

In [ ]:
# Cell 19G: Benchmark fixed one-beam and four-beam translation
def mt_run_fixed_baselines():
    if not globals().get('MT_DATA_READY', False):
        raise RuntimeError('Run Cell 19E successfully first.')
    if not callable(globals().get('load_mt')):
        raise RuntimeError('Run Cell 19B first.')
    if MT_DATA_RECORD['settings']['split'] != 'dev':
        raise ValueError('This development benchmark must only use dev.')
    if MT_BENCH_REPEATS < 1 or MT_BENCH_WARMUPS < 1:
        raise ValueError('Repeats and warmups must be positive.')

    # Ensure both directions still use exactly the saved references and IDs.
    if mt_data_hash(MT_DATA_RECORD['pairs']) != MT_DATA_RECORD['sha256']:
        raise ValueError('The reference data changed; rerun Cell 19E.')

    for source, target in (('en', 'zh'), ('zh', 'en')):
        expected = [
            dict(id=r['id'], source=r[source], reference=r[target])
            for r in MT_DATA_RECORD['pairs']
        ]
        if not expected or MT_DEV[source] != expected:
            raise ValueError('Benchmark rows differ from the saved subset.')

    folder = Path(RESULTS_DIR) / 'mt_baselines' / str(time.time_ns())
    folder.mkdir(parents=True, exist_ok=False)
    print('Results folder:', folder)

    summaries = []
    release_mt()

    try:
        for language in ('en', 'zh'):
            loaded = load_mt(language)
            model = loaded['model']
            device = next(model.parameters()).device
            rows = MT_DEV[language]

            # Fail clearly on overlong inputs; never truncate or drop rows.
            limit = min(
                MT_BENCH_INPUT_LIMIT, model.config.max_position_embeddings
            )
            for row in rows:
                prepared = row['source'].strip()
                if language == 'en':
                    prepared = f'{MT_ZH_TAG} {prepared}'

                count = len(
                    loaded['tokenizer'](
                        prepared, truncation=False
                    )['input_ids']
                )
                if count > limit:
                    raise ValueError(
                        f"{language}/{row['id']}: "
                        f"{count} tokens exceeds {limit}."
                    )

            for beams in MT_BENCH_BEAMS:
                for i in range(MT_BENCH_WARMUPS):
                    mt_decode_loaded(
                        rows[i % len(rows)]['source'],
                        language, loaded, beams
                    )

                if device.type == 'cuda':
                    torch.cuda.reset_peak_memory_stats(device)

                predictions = []
                timings = []

                for i, row in enumerate(rows, 1):
                    trials = [
                        mt_decode_loaded(
                            row['source'], language, loaded, beams
                        )
                        for _ in range(MT_BENCH_REPEATS)
                    ]
                    first = trials[0]
                    times = [trial['seconds'] for trial in trials]
                    timings.extend(times)

                    predictions.append(dict(
                        id=row['id'],
                        source=row['source'],
                        reference=row['reference'],
                        prediction=first['prediction'],
                        latency_seconds=times,
                        input_tokens=first['input_tokens'],
                        output_tokens=first['output_tokens'],
                        hit_output_limit=any(
                            t['hit_output_limit'] for t in trials
                        ),
                        repeat_output_changed=any(
                            t['prediction'] != first['prediction']
                            for t in trials[1:]
                        ),
                    ))

                    if i % 25 == 0 or i == len(rows):
                        print(
                            f'{language} | beam {beams} | '
                            f'{i}/{len(rows)} sentences'
                        )

                # Score each sentence once; repeated calls are only for timing.
                refs = [r['reference'] for r in predictions]
                hyps = [r['prediction'] for r in predictions]

                bleu = BLEU(
                    tokenize='zh' if language == 'en' else '13a'
                )
                chrf = CHRF(char_order=6, word_order=0, beta=2)

                summary = dict(
                    direction='en→zh' if language == 'en' else 'zh→en',
                    beams=beams,
                    samples=len(rows),
                    BLEU=bleu.corpus_score(hyps, [refs]).score,
                    chrF=chrf.corpus_score(hyps, [refs]).score,
                    median_seconds=float(np.median(timings)),
                    p95_seconds=float(np.percentile(timings, 95)),
                    timing_calls=len(timings),
                    empty_outputs=sum(not h for h in hyps),
                    output_limit_hits=sum(
                        r['hit_output_limit'] for r in predictions
                    ),
                    changed_on_repeat=sum(
                        r['repeat_output_changed'] for r in predictions
                    ),
                )

                record = dict(
                    schema_version=1,
                    dataset=MT_DATA_RECORD['settings'],
                    reference_sha256=MT_DATA_RECORD['sha256'],
                    model_id=loaded['model_id'],
                    revision=loaded['revision'],
                    precision=str(next(model.parameters()).dtype),
                    device=str(device),
                    hardware=(
                        torch.cuda.get_device_name(device)
                        if device.type == 'cuda' else 'CPU'
                    ),
                    versions=dict(
                        python=sys.version.split()[0],
                        torch=str(torch.__version__),
                        transformers=transformers.__version__,
                        sacrebleu=sacrebleu.__version__,
                    ),
                    cpu_threads=torch.get_num_threads(),
                    generation_defaults=model.generation_config.to_dict(),
                    decoding=dict(
                        num_beams=beams,
                        num_return_sequences=1,
                        max_new_tokens=MT_BENCH_OUTPUT_LIMIT,
                        max_length=None,
                        do_sample=False,
                        early_stopping=(beams > 1),
                        length_penalty=1.0,
                        use_cache=True,
                    ),
                    effective_input_limit=limit,
                    chinese_target_tag=MT_ZH_TAG,
                    timing=dict(
                        repeats=MT_BENCH_REPEATS,
                        warmups=MT_BENCH_WARMUPS,
                        batch_size=1,
                        includes=(
                            'tokenization, device transfer, '
                            'generation, decoding'
                        ),
                        excludes=(
                            'model loading, warmup, metrics and file saving'
                        ),
                    ),
                    peak_gpu_allocated_mb=(
                        torch.cuda.max_memory_allocated(device) / 1024**2
                        if device.type == 'cuda' else None
                    ),
                    metric_signatures=dict(
                        BLEU=str(bleu.get_signature()),
                        chrF=str(chrf.get_signature()),
                    ),
                    COMET=None,
                    COMET_status='Pending separate scoring step',
                    summary=summary,
                    predictions=predictions,
                )

                destination = folder / f'{language}_beam{beams}.json'
                destination.write_text(
                    json.dumps(record, ensure_ascii=False, indent=2),
                    encoding='utf-8',
                )
                summaries.append(summary)
                print('Saved:', destination.name)

    finally:
        release_mt()

    table = pd.DataFrame(summaries)
    table.to_csv(folder / 'summary.csv', index=False)
    display(table.round(3))

    print('Development benchmark complete. Final-test data was not used.')
    return table, folder


MT_BASELINE_TABLE, MT_BASELINE_DIR = None, None
MT_BASELINE_TABLE, MT_BASELINE_DIR = mt_run_fixed_baselines()

In [ ]:
# Cell 19H: Compare beam settings sentence by sentence on the saved dev subset
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from sacrebleu.metrics import CHRF

mt_pair_rows = []
mt_sentence_chrf = CHRF(char_order=6, word_order=0, beta=2)

for language in ('en', 'zh'):
    records = [
        json.loads(
            (Path(MT_BASELINE_DIR) / f'{language}_beam{beams}.json')
            .read_text(encoding='utf-8')
        )
        for beams in (1, 4)
    ]
    fast, slow = records

    # Pair the same sentences; keep this analysis restricted to dev.
    assert fast['dataset'] == slow['dataset']
    assert fast['dataset']['split'] == 'dev'

    for first, second in zip(
        fast['predictions'], slow['predictions'], strict=True
    ):
        assert all(
            first[k] == second[k]
            for k in ('id', 'source', 'reference')
        )

        # References are used for analysis, never for live retry decisions.
        scores = [
            mt_sentence_chrf.sentence_score(
                row['prediction'], [row['reference']]
            ).score
            for row in (first, second)
        ]

        mt_pair_rows.append(dict(
            direction='en→zh' if language == 'en' else 'zh→en',
            id=first['id'],
            beam1_chrF=scores[0],
            beam4_chrF=scores[1],
            chrF_change=scores[1] - scores[0],
            same_output=first['prediction'] == second['prediction'],
            source=first['source'],
            reference=first['reference'],
            beam1=first['prediction'],
            beam4=second['prediction'],
        ))

MT_PAIR_TABLE = pd.DataFrame(mt_pair_rows)

MT_PAIR_SUMMARY = MT_PAIR_TABLE.groupby('direction', sort=False).agg(
    samples=('id', 'size'),
    higher_chrF=('chrF_change', lambda x: int((x > 1e-6).sum())),
    lower_chrF=('chrF_change', lambda x: int((x < -1e-6).sum())),
    tied_chrF=('chrF_change', lambda x: int((x.abs() <= 1e-6).sum())),
    identical_outputs=('same_output', 'sum'),
)
display(MT_PAIR_SUMMARY)

review_columns = [
    'id', 'chrF_change', 'source', 'beam1', 'beam4', 'reference'
]

for direction, group in MT_PAIR_TABLE.groupby('direction', sort=False):
    print(f'\n{direction}: two largest chrF increases and decreases')
    examples = pd.concat([
        group.nlargest(2, 'chrF_change'),
        group.nsmallest(2, 'chrF_change'),
    ])
    with pd.option_context('display.max_colwidth', 100):
        display(examples[review_columns].round(3))

# Save full text so later inspection does not require generating it again.
mt_pair_path = Path(MT_BASELINE_DIR) / 'sentence_comparison.csv'
MT_PAIR_TABLE.to_csv(mt_pair_path, index=False, encoding='utf-8-sig')

print('\nSaved:', mt_pair_path)
print('Higher chrF means closer to the reference, not guaranteed correctness.')

In [ ]:
# Cell 19I: Reference-free checks for translation retry candidates
import sys
import subprocess

# This parser requires the older emoji API.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'recognizers-text-number==1.0.2a2',
    'emoji==1.7.0',
])

import re
import unicodedata
from decimal import Decimal
from recognizers_number import (
    recognize_number, recognize_ordinal, recognize_percentage,
)

MT_CHECK_VERSION = 'numeric_repetition_v1'
MT_CULTURE = {'en': 'en-us', 'zh': 'zh-cn'}
MT_MONTHS = {
    name: str(i)
    for i, name in enumerate(
        ('January February March April May June July August '
         'September October November December').split(), 1
    )
}

# Convert common length, mass and voltage units to shared units.
MT_UNIT_RULES = [
    (r'(?<![a-z])(?:kilometers?|kilometres?|km)(?![a-z])|公里|千米',
     'KM', 'm', '1000'),
    (r'(?<![a-z])(?:centimeters?|centimetres?|cm)(?![a-z])|厘米',
     'CM', 'm', '0.01'),
    (r'(?<![a-z])(?:millimeters?|millimetres?|mm)(?![a-z])|毫米',
     'MM', 'm', '0.001'),
    (r'(?<![a-z])(?:meters?|metres?|m)(?![a-z])|米',
     'M', 'm', '1'),
    (r'(?<![a-z])(?:kilograms?|kg)(?![a-z])|公斤|千克',
     'KG', 'g', '1000'),
    (r'(?<![a-z])(?:milligrams?|mg)(?![a-z])|毫克',
     'MG', 'g', '0.001'),
    (r'(?<![a-z])(?:grams?|g)(?![a-z])|克',
     'G', 'g', '1'),
    (r'(?<![a-z])(?:volts?|v)(?![a-z])|伏特|伏',
     'V', 'V', '1'),
]
MT_UNIT_VALUES = {
    code: (unit, Decimal(scale))
    for _, code, unit, scale in MT_UNIT_RULES
}


def mt_quantities(text, language):
    """Extract normalized quantities; this is not semantic validation."""
    text = unicodedata.normalize('NFKC', text).replace('−', '-')
    text = re.sub(r'\bB\.C\.(?:E\.?)?', 'BCE', text, flags=re.I)

    if language == 'en':
        text = re.sub(
            r'(?<=\d)(hundred|thousand|million|billion)\b',
            r' \1', text, flags=re.I,
        )

        # Handle common month/day forms without changing modal "may".
        months = '|'.join(name for name in MT_MONTHS if name != 'May')
        text = re.sub(
            r'\b(' + months + r')\b',
            lambda m: MT_MONTHS[m[0]], text,
        )
        text = re.sub(r'(\d+)(?:st|nd|rd|th)\b', r'\1', text)
        text = re.sub(r'\bMay(?=\s+\d)', '5', text)
        text = re.sub(r'(?<=\d)\s+May\b', ' 5', text)

    # 9,000 and 9 000 both become 9000, including before Chinese 万/亿.
    text = re.sub(
        r'(?<!\d)\d{1,3}(?:[, ]\d{3})+(?!\d)',
        lambda m: re.sub(r'[, ]', '', m.group()), text,
    )
    for pattern, code, _, _ in MT_UNIT_RULES:
        text = re.sub(pattern, f' <{code}> ', text, flags=re.I)

    culture = MT_CULTURE[language]
    matches = (
        [('percent', r) for r in recognize_percentage(text, culture)]
        + [('number', r) for r in recognize_number(text, culture)]
        + [('ordinal', r) for r in recognize_ordinal(text, culture)]
    )

    # Prefer complete expressions, avoiding counting both "50%" and "50".
    matches.sort(key=lambda item: -(item[1].end - item[1].start))
    quantities, covered = set(), set()

    for kind, result in matches:
        span = set(range(result.start, result.end + 1))
        if covered & span:
            continue
        covered.update(span)

        # Alphanumeric identifiers are outside this quantity check.
        if (
            re.search(r'[A-Za-z]$', text[:result.start])
            or re.match(r'[A-Za-z]', text[result.end + 1:])
        ):
            continue

        value = Decimal(result.resolution['value'].rstrip('%'))

        if kind == 'ordinal' and (
            value <= 0
            or re.search(r'\bper\s+$', text[:result.start], re.I)
        ):
            continue

        # Avoid treating isolated numeral characters inside words as numbers.
        if (
            language == 'zh'
            and not re.search(r'\d', result.text)
            and len(result.text) == 1
            and text.strip() != result.text
            and not re.match(
                r'\s*(?:<|[人名个位只张本条岁倍件台座种例家枚边面方时年月日天])',
                text[result.end + 1:],
            )
        ):
            continue

        unit = ''
        if kind == 'percent':
            value /= 100
        else:
            suffix = re.match(
                r'\s*(?:a\s+)?<([A-Z]+)>',
                text[result.end + 1:],
            )
            if suffix:
                unit, scale = MT_UNIT_VALUES[suffix[1]]
                value *= scale

        # Unqualified "one"/一 often acts as an article rather than a quantity.
        if value == 1 and not unit:
            continue

        # Repeating a quantity in different wording is not itself an error.
        quantities.add((format(value.normalize(), 'f'), unit))

    return quantities


def mt_has_repetition(text):
    # Use Chinese characters and English words to find repeated phrases.
    tokens = re.findall(
        r'[\u4e00-\u9fff]|[a-z0-9]+',
        unicodedata.normalize('NFKC', text).casefold(),
    )

    for width in range(1, min(8, len(tokens) // 3) + 1):
        for start in range(len(tokens) - 3 * width + 1):
            phrase = tokens[start:start + width]
            if (
                tokens[start:start + 3 * width] == phrase * 3
                and sum(map(len, phrase)) * 3 >= 6
            ):
                return True

    return False


def mt_check_translation(source, prediction, source_language):
    """Use source text and model output only; never reference translations."""
    target_language = {'en': 'zh', 'zh': 'en'}[source_language]

    before = mt_quantities(source, source_language)
    after = mt_quantities(prediction, target_language)
    missing, added = before - after, after - before

    flags = []
    if not prediction.strip():
        flags.append('empty_output')

    if mt_has_repetition(prediction) and not mt_has_repetition(source):
        flags.append('repetition')

    if missing or added:
        flags.append('quantity_mismatch')

    return dict(
        flags=flags,
        flagged=bool(flags),
        missing_quantities=sorted(missing),
        added_quantities=sorted(added),
    )


print('Translation checks ready.')

In [ ]:
# Cell 19J: Verify the checks and inspect flags on saved dev predictions
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

mt_check_examples = [
    ('1.5 million people', '150万人', 'en', []),
    ('0.5 kg', '500克', 'en', []),
    ('twelve volts', '十二伏特', 'en', []),
    ('12V', '12伏', 'en', []),
    ('50%', '百分之五十', 'en', []),
    ('October 7', '10月7日', 'en', []),
    ('He wrote a book.', '他写了一本书。', 'en', []),
    ('预计有9000人参加。', '900,000 people are expected.',
     'zh', ['quantity_mismatch']),
    ('Voltage is 12 V.', '电压是21伏。',
     'en', ['quantity_mismatch']),
    ('Please close the door.', '', 'en', ['empty_output']),
    ('Please close the door.', '请关门请关门请关门',
     'en', ['repetition']),
]

for source, prediction, language, expected in mt_check_examples:
    actual = mt_check_translation(source, prediction, language)['flags']
    assert actual == expected, (source, prediction, actual, expected)

print(f'{len(mt_check_examples)} checks passed.')

mt_check_rows = []

for row in MT_PAIR_TABLE.to_dict('records'):
    language = row['direction'].split('→')[0]

    for beams in (1, 4):
        prediction = row[f'beam{beams}']

        # The check receives no reference text or chrF score.
        result = mt_check_translation(
            row['source'], prediction, language
        )
        mt_check_rows.append(dict(
            direction=row['direction'],
            id=row['id'],
            beams=beams,
            source=row['source'],
            prediction=prediction,
            **result,
        ))

MT_CHECK_TABLE = pd.DataFrame(mt_check_rows)

MT_CHECK_SUMMARY = MT_CHECK_TABLE.groupby(['direction', 'beams']).agg(
    samples=('id', 'size'),
    flagged=('flagged', 'sum'),
    empty=('flags', lambda x: sum('empty_output' in f for f in x)),
    repetition=('flags', lambda x: sum('repetition' in f for f in x)),
    quantities=('flags', lambda x: sum('quantity_mismatch' in f for f in x)),
)
display(MT_CHECK_SUMMARY)

# Inspect a few fast-output flags before using them to trigger retries.
for direction, group in MT_CHECK_TABLE.groupby('direction', sort=False):
    print(f'\n{direction}: flagged beam-1 examples')

    examples = group[
        group['flagged'] & group['beams'].eq(1)
    ].head(5)

    with pd.option_context('display.max_colwidth', 100):
        display(examples[[
            'id', 'source', 'prediction', 'flags',
            'missing_quantities', 'added_quantities',
        ]])

mt_check_path = Path(MT_BASELINE_DIR) / 'quality_checks_v1.json'
mt_check_path.write_text(
    json.dumps(
        dict(check_version=MT_CHECK_VERSION, records=mt_check_rows),
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)

print('\nSaved:', mt_check_path)
print('Flags are suspected problems, not measured translation error rates.')

In [ ]:
# Cell 20 (LAST): Upload, camera capture, and live line transcription
import base64, io
from google.colab import output
from IPython.display import JSON, Javascript, display


def interactive_ocr(data_url, lang, version, compare=False):
    try:
        if lang not in BASE:
            raise ValueError('Choose English or Chinese.')

        raw = base64.b64decode(data_url.split(',', 1)[1])
        with Image.open(io.BytesIO(raw)) as im:
            image = im.convert('RGB')

        versions = (
            list(model_sources(lang)) if compare else [version]
        )
        results = []

        for selected in versions:
            model, processor = load_ocr(lang, selected)

            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            started = time.perf_counter()

            text = transcribe(image, lang, model, processor)

            if DEVICE == 'cuda':
                torch.cuda.synchronize()

            results.append(dict(
                model=selected,
                text=text,
                seconds=round(
                    time.perf_counter() - started, 2
                ),
            ))

        return JSON({'results': results})

    except Exception as e:
        return JSON({'error': str(e)})


output.register_callback(
    'curated.transcribe', interactive_ocr
)

options = {
    lang: list(model_sources(lang)) for lang in BASE
}
for lang, available in options.items():
    print(
        lang, '| available:', available,
        '| unavailable:',
        [
            x for x in ('baseline', 'public', 'adapted')
            if x not in available
        ],
    )

display(Javascript(r"""
(() => {
  const available = __OPTIONS__;
  const box = document.createElement('div');

  box.innerHTML = `
    <p>Use one cropped handwritten line.
       Camera: place it inside the green guide.</p>
    <select class="language">
      <option value="en">English</option>
      <option value="zh">Chinese</option>
    </select>
    <select class="model"></select>
    <input class="upload" type="file" accept="image/*">

    <div style="position:relative;max-width:640px">
      <video playsinline muted
             style="width:100%;display:block"></video>
      <div style="position:absolute;left:8%;top:43%;
                  width:87%;height:14%;border:3px solid lime;
                  box-sizing:border-box;pointer-events:none"></div>
    </div>

    <button class="camera">Open camera</button>
    <button class="capture">Capture line</button>
    <button class="compare">Compare last image</button>
    <button class="start">Start live OCR</button>
    <button class="stop">Stop camera</button>

    <img class="preview"
         style="display:none;max-width:640px;max-height:180px">
    <pre class="status"
         style="white-space:pre-wrap">Ready.</pre>
  `;
  document.body.appendChild(box);

  const get = name => box.querySelector('.' + name);
  const language = get('language');
  const model = get('model');
  const video = box.querySelector('video');
  const status = get('status');

  let stream = null;
  let running = false;
  let busy = false;
  let lastImage = null;

  function resize() {
    google.colab.output.setIframeHeight(
      document.documentElement.scrollHeight, true
    );
  }

  function models() {
    model.replaceChildren();
    for (const key of ['baseline', 'public', 'adapted']) {
      const option = new Option(key, key);
      option.disabled = !available[language.value].includes(key);
      if (option.disabled) option.text += ' (unavailable)';
      model.add(option);
    }
    model.value = available[language.value].slice(-1)[0];
  }

  function controls() {
    for (const name of [
      'language', 'model', 'upload', 'camera', 'start'
    ]) {
      get(name).disabled = busy || running;
    }
    get('camera').disabled ||= !!stream;
    get('capture').disabled = busy || running || !stream;
    get('compare').disabled = busy || running || !lastImage;
    get('stop').disabled = !stream && !running;
  }

  function stop() {
    running = false;
    if (stream) {
      stream.getTracks().forEach(track => track.stop());
    }
    stream = null;
    video.srcObject = null;
    controls();
  }

  async function camera() {
    if (!stream) {
      stream = await navigator.mediaDevices.getUserMedia({
        video: {
          width: {ideal: 1920},
          height: {ideal: 1080},
          facingMode: {ideal: 'environment'}
        },
        audio: false
      });
      video.srcObject = stream;
      await video.play();
      resize();
    }
  }

  function crop() {
    if (!video.videoWidth) {
      throw new Error('Camera is not ready.');
    }

    const canvas = document.createElement('canvas');
    canvas.width = Math.floor(video.videoWidth * .87);
    canvas.height = Math.floor(video.videoHeight * .14);

    canvas.getContext('2d').drawImage(
      video,
      Math.floor(video.videoWidth * .08),
      Math.floor(video.videoHeight * .43),
      canvas.width, canvas.height,
      0, 0, canvas.width, canvas.height
    );

    return canvas.toDataURL('image/jpeg', .95);
  }

  async function recognise(data, compare=false) {
    lastImage = data;
    get('preview').src = data;
    get('preview').style.display = 'block';

    status.textContent =
      'Transcribing (first use may load the model)...';
    resize();

    const response = await google.colab.kernel.invokeFunction(
      'curated.transcribe',
      [data, language.value, model.value, compare],
      {}
    );
    const result = response.data['application/json'];

    if (result.error) throw new Error(result.error);

    status.textContent = result.results.map(
      r => `${r.model}: ${r.text || '(empty)'}\n` +
           `OCR: ${r.seconds}s (loading excluded)`
    ).join('\n\n');

    resize();
  }

  async function action(fn) {
    if (busy || running) return;
    busy = true;
    controls();

    try {
      await fn();
    } catch (e) {
      stop();
      status.textContent = e.message;
    } finally {
      busy = false;
      controls();
      resize();
    }
  }

  get('upload').onchange = () => action(async () => {
    const file = get('upload').files[0];
    if (!file) return;

    const data = await new Promise((resolve, reject) => {
      const reader = new FileReader();
      reader.onload = () => resolve(reader.result);
      reader.onerror = () => reject(
        new Error('Cannot read image.')
      );
      reader.readAsDataURL(file);
    });

    await recognise(data);
  });

  get('camera').onclick = () => action(camera);
  get('capture').onclick = () => action(
    () => recognise(crop())
  );
  get('compare').onclick = () => action(
    () => recognise(lastImage, true)
  );

  get('start').onclick = () => action(async () => {
    await camera();
    running = true;
    controls();

    const started = Date.now();

    while (running && Date.now() - started < 60000) {
      const tick = performance.now();
      await recognise(crop());

      if (!running) break;

      await new Promise(resolve => setTimeout(
        resolve,
        Math.max(100, 1500 - (performance.now() - tick))
      ));
    }

    stop();
    status.textContent +=
      '\nLive OCR stopped (provisional transcription).';
  });

  get('stop').onclick = () => {
    stop();
    status.textContent += '\nCamera stopped.';
  };

  window.addEventListener('pagehide', stop);
  language.onchange = models;

  models();
  controls();
  resize();
})();
""".replace('__OPTIONS__', json.dumps(options))))